# Portfolio Construction

## Research question

Which sleeve-allocation design best balances performance, diversification, exposure control, and implementation stability?

This notebook first reconciles the drift-aware backtester, then compares equal-weight, composite-score, exposure-controlled, dynamic risk-balanced, and pure inverse-volatility portfolios before benchmarking against SPY.

## 1. Backtest-engine audit

### 1.1 Setup

In [1]:
import numpy as np
import pandas as pd

from alpha_research.backtest import (
    BacktestConfig,
    run_long_short_backtest,
    run_target_weight_backtest,
    summarise_backtest,
)
from alpha_research.config.paths import PROCESSED_DATA_DIR
from alpha_research.data_loader import load_parquet
from alpha_research.portfolio import build_factor_target_weights


panel = load_parquet(
    PROCESSED_DATA_DIR / "factor_panel.parquet"
)

factor_columns = {
    "12-1 Momentum": "mom_12_1m_z",
    "Realised Volatility": "realised_vol_63_z",
}

config = BacktestConfig(
    rebalance_frequency=5,
    quantiles=5,
    long_quantile=5,
    short_quantile=1,
    long_gross=1.0,
    short_gross=1.0,
    transaction_cost_bps=10.0,
    min_observations=30,
    rebalance_offset=0,
)

### 1.2 Run both engines

In [2]:
audit_results = {}

return_panel = panel[
    ["date", "ticker", "forward_ret_1d"]
].copy()

for factor_name, factor_column in factor_columns.items():
    target_weights = build_factor_target_weights(
        panel=panel,
        factor_column=factor_column,
        return_column="forward_ret_1d",
        config=config,
    )

    legacy_daily, legacy_holdings = run_long_short_backtest(
        panel=panel,
        factor_column=factor_column,
        return_column="forward_ret_1d",
        config=config,
    )

    drift_daily, drift_holdings = run_target_weight_backtest(
        return_panel=return_panel,
        target_weights=target_weights,
        return_column="forward_ret_1d",
        transaction_cost_bps=config.transaction_cost_bps,
    )

    audit_results[factor_name] = {
        "targets": target_weights,
        "legacy_daily": legacy_daily,
        "legacy_holdings": legacy_holdings,
        "drift_daily": drift_daily,
        "drift_holdings": drift_holdings,
    }

### 1.3 Verify the results

Both engines should match the target weights exactly on rebalance dates.

In [3]:
def maximum_weight_difference(
    left: pd.DataFrame,
    right: pd.DataFrame,
) -> float:
    comparison = left[["date", "ticker", "weight"]].merge(
        right[["date", "ticker", "weight"]],
        on=["date", "ticker"],
        how="outer",
        suffixes=("_left", "_right"),
    )

    comparison[["weight_left", "weight_right"]] = comparison[
        ["weight_left", "weight_right"]
    ].fillna(0.0)

    return float((comparison["weight_left"] - comparison["weight_right"]).abs().max())


audit_check_rows = []

for factor_name, result in audit_results.items():
    targets = result["targets"]
    target_dates = targets["date"].unique()

    legacy_rebalance_holdings = result["legacy_holdings"].loc[
        lambda df: df["date"].isin(target_dates),
        ["date", "ticker", "weight"],
    ]

    drift_rebalance_holdings = result["drift_holdings"].loc[
        lambda df: df["date"].isin(target_dates),
        ["date", "ticker", "weight"],
    ]

    daily_comparison = result["legacy_daily"][
        ["date", "is_rebalance", "gross_return"]
    ].merge(
        result["drift_daily"][["date", "is_rebalance", "gross_return"]],
        on="date",
        how="inner",
        suffixes=("_legacy", "_drift"),
        validate="one_to_one",
    )

    gross_difference = (
        daily_comparison["gross_return_drift"] - daily_comparison["gross_return_legacy"]
    ).abs()

    rebalance_mask = daily_comparison["is_rebalance_legacy"]

    audit_check_rows.append(
        {
            "factor": factor_name,
            "same_daily_dates": (
                len(daily_comparison)
                == len(result["legacy_daily"])
                == len(result["drift_daily"])
            ),
            "rebalance_dates": len(target_dates),
            "max_legacy_target_mismatch": (
                maximum_weight_difference(
                    legacy_rebalance_holdings,
                    targets,
                )
            ),
            "max_drift_target_mismatch": (
                maximum_weight_difference(
                    drift_rebalance_holdings,
                    targets,
                )
            ),
            "max_rebalance_gross_return_difference": (
                gross_difference.loc[rebalance_mask].max()
            ),
            "mean_non_rebalance_gross_return_difference": (
                gross_difference.loc[~rebalance_mask].mean()
            ),
        }
    )

audit_checks = pd.DataFrame(audit_check_rows).set_index("factor")

audit_checks

,same_daily_dates,rebalance_dates,max_legacy_target_mismatch,max_drift_target_mismatch,max_rebalance_gross_return_difference,mean_non_rebalance_gross_return_difference
factor,,,,,,
12-1 Momentum,True,578,0.0,0.0,0.0,0.000349
Realised Volatility,True,578,0.0,0.0,0.0,0.000262


#### 1.3.1 Compare economic results

In [4]:
summary_rows = []

for factor_name, result in audit_results.items():
    for engine_name, daily in {
        "Legacy": result["legacy_daily"],
        "Drift-aware": result["drift_daily"],
    }.items():
        gross = summarise_backtest(
            daily,
            return_column="gross_return",
        ).iloc[0]

        net = summarise_backtest(
            daily,
            return_column="net_return",
        ).iloc[0]

        summary_rows.append(
            {
                "factor": factor_name,
                "engine": engine_name,
                "gross_total_return": gross["total_return"],
                "net_total_return": net["total_return"],
                "gross_annualised_return": (gross["annualised_return"]),
                "net_annualised_return": (net["annualised_return"]),
                "net_annualised_volatility": (net["annualised_volatility"]),
                "gross_sharpe": gross["sharpe_ratio"],
                "net_sharpe": net["sharpe_ratio"],
                "average_rebalance_turnover": (net["average_rebalance_turnover"]),
                "total_transaction_cost": (net["total_transaction_cost"]),
            }
        )

audit_summary = (
    pd.DataFrame(summary_rows).sort_values(["factor", "engine"]).reset_index(drop=True)
)

audit_summary.round(4)

,factor,engine,gross_total_return,net_total_return,gross_annualised_return,net_annualised_return,net_annualised_volatility,gross_sharpe,net_sharpe,average_rebalance_turnover,total_transaction_cost
0,12-1 Momentum,Drift-aware,0.4804,0.1038,0.0348,0.0086,0.2119,0.2683,0.1475,0.5080,0.2936
1,12-1 Momentum,Legacy,0.5957,0.2379,0.0416,0.0188,0.2113,0.2995,0.1947,0.4393,0.2539
2,Realised Volatility,Drift-aware,5.5496,4.1786,0.1781,0.1542,0.2395,0.8042,0.7185,0.4063,0.2348
3,Realised Volatility,Legacy,5.5559,4.3711,0.1782,0.1579,0.2399,0.8036,0.7310,0.3448,0.1993


#### 1.3.2 Isolate the change

In [5]:
metric_columns = [
    "gross_annualised_return",
    "net_annualised_return",
    "net_annualised_volatility",
    "gross_sharpe",
    "net_sharpe",
    "average_rebalance_turnover",
    "total_transaction_cost",
]

legacy_summary = audit_summary.loc[audit_summary["engine"] == "Legacy"].set_index(
    "factor"
)

drift_summary = audit_summary.loc[audit_summary["engine"] == "Drift-aware"].set_index(
    "factor"
)

audit_deltas = drift_summary[metric_columns] - legacy_summary[metric_columns]

audit_deltas.columns = [f"change_in_{column}" for column in audit_deltas.columns]

audit_deltas.round(4)

,change_in_gross_annualised_return,change_in_net_annualised_return,change_in_net_annualised_volatility,change_in_gross_sharpe,change_in_net_sharpe,change_in_average_rebalance_turnover,change_in_total_transaction_cost
factor,,,,,,,
12-1 Momentum,-0.0068,-0.0101,0.0006,-0.0312,-0.0472,0.0687,0.0397
Realised Volatility,-0.0001,-0.0037,-0.0004,0.0006,-0.0125,0.0615,0.0355


### 1.4 Engine-audit conclusion

The drift-aware engine reproduces the legacy target portfolios exactly on rebalance dates. Rebalance-day gross returns also match exactly, confirming that the factor signals, stock selection, and target-weight construction are unchanged.

Between rebalances, the drift-aware engine carries forward holdings whose weights evolve with asset returns. This produces non-zero return differences relative to the legacy constant-weight convention and increases measured rebalance turnover.

The legacy implementation understated average rebalance turnover by approximately 0.069 for momentum and 0.062 for realised volatility. After realistic drift accounting, momentum's net Sharpe declines from approximately 0.195 to 0.148, while realised volatility's net Sharpe declines from 0.731 to 0.719.

All subsequent portfolio experiments will therefore use the drift-aware target-weight engine. Legacy results will be retained only as historical benchmarks.

## 2. Equal-weight sleeve allocation

A 50/50 combination of momentum and realised volatility, independent sleeves

In [6]:
from alpha_research.portfolio import (
    build_factor_target_weights,
    combine_sleeve_target_weights,
)

In [7]:
momentum_targets = audit_results["12-1 Momentum"]["targets"]

volatility_targets = audit_results["Realised Volatility"]["targets"]

combined_targets = combine_sleeve_target_weights(
    sleeve_targets={
        "Momentum": momentum_targets,
        "Realised Volatility": volatility_targets,
    },
    sleeve_allocations={
        "Momentum": 0.5,
        "Realised Volatility": 0.5,
    },
)

combined_daily, combined_holdings = run_target_weight_backtest(
    return_panel=return_panel,
    target_weights=combined_targets,
    return_column="forward_ret_1d",
    transaction_cost_bps=config.transaction_cost_bps,
)

### 2.1 Examine natural netting

In [8]:
combined_target_exposure = (
    combined_targets.assign(
        long_weight=lambda df: df["weight"].clip(lower=0.0),
        short_weight=lambda df: -df["weight"].clip(upper=0.0),
        active_position=lambda df: df["weight"].ne(0.0).astype(int),
    )
    .groupby("date")
    .agg(
        long_exposure=("long_weight", "sum"),
        short_exposure=("short_weight", "sum"),
        active_positions=("active_position", "sum"),
    )
)

combined_target_exposure["gross_exposure"] = (
    combined_target_exposure["long_exposure"]
    + combined_target_exposure["short_exposure"]
)

combined_target_exposure["net_exposure"] = (
    combined_target_exposure["long_exposure"]
    - combined_target_exposure["short_exposure"]
)

combined_target_exposure[
    [
        "long_exposure",
        "short_exposure",
        "gross_exposure",
        "net_exposure",
        "active_positions",
    ]
].agg(["mean", "std", "min", "max"]).round(4)

,long_exposure,short_exposure,gross_exposure,net_exposure,active_positions
mean,0.7352,0.7352,1.4704,0.0,49.9204
std,0.1905,0.1905,0.3810,0.0,10.8236
min,0.0000,0.0000,0.0000,0.0,0.0000
max,1.0000,1.0000,2.0000,0.0,66.0000


### 2.2 Compare the three portfolios

In [9]:
baseline_daily = {
    "Momentum": audit_results["12-1 Momentum"]["drift_daily"],
    "Realised Volatility": audit_results["Realised Volatility"]["drift_daily"],
    "50/50 Independent Sleeves": combined_daily,
}

baseline_rows = []

for portfolio_name, daily in baseline_daily.items():
    gross = summarise_backtest(
        daily,
        return_column="gross_return",
    ).iloc[0]

    net = summarise_backtest(
        daily,
        return_column="net_return",
    ).iloc[0]

    baseline_rows.append(
        {
            "portfolio": portfolio_name,
            "gross_annualised_return": gross["annualised_return"],
            "net_annualised_return": net["annualised_return"],
            "net_annualised_volatility": net["annualised_volatility"],
            "gross_sharpe": gross["sharpe_ratio"],
            "net_sharpe": net["sharpe_ratio"],
            "max_drawdown": net["max_drawdown"],
            "average_rebalance_turnover": net["average_rebalance_turnover"],
            "total_transaction_cost": net["total_transaction_cost"],
            "average_gross_exposure": daily["gross_exposure"].mean(),
        }
    )

baseline_summary = pd.DataFrame(baseline_rows).set_index("portfolio")

baseline_summary.round(4)

,gross_annualised_return,net_annualised_return,net_annualised_volatility,gross_sharpe,net_sharpe,max_drawdown,average_rebalance_turnover,total_transaction_cost,average_gross_exposure
portfolio,,,,,,,,,
Momentum,0.0348,0.0086,0.2119,0.2683,0.1475,-0.5096,0.5080,0.2936,1.8262
Realised Volatility,0.1781,0.1542,0.2395,0.8042,0.7185,-0.4585,0.4063,0.2348,1.9556
50/50 Independent Sleeves,0.1213,0.0982,0.1616,0.7900,0.6610,-0.2566,0.4130,0.2387,1.4712


In [10]:
gross_return_comparison = pd.concat(
    {
        name: daily.set_index("date")["gross_return"]
        for name, daily in baseline_daily.items()
    },
    axis=1,
)

gross_return_comparison.corr().round(4)

,Momentum,Realised Volatility,50/50 Independent Sleeves
Momentum,1.0000,0.0225,0.6696
Realised Volatility,0.0225,1.0000,0.7572
50/50 Independent Sleeves,0.6696,0.7572,1.0000


### 2.3 Findings: 50/50 independent factor sleeves

The independent-sleeve portfolio allocates 50% of notional capital to the momentum portfolio and 50% to the realised-volatility portfolio, then nets their stock-level target weights.

The standalone factor returns have very low correlation (approximately 0.02), providing meaningful diversification. Factor disagreement reduces average gross exposure to approximately 1.47, while dollar neutrality is preserved.

The combined portfolio produces a net annualised return of 9.82%, volatility of 16.16%, and a net Sharpe ratio of 0.661. Its maximum drawdown of -25.66% is substantially smaller than the drawdowns of either standalone factor.

The portfolio does not exceed the realised-volatility factor's standalone Sharpe ratio, but it delivers a materially smoother risk profile through diversification and natural position netting.

## 3. Composite-score portfolio

Composite score = average of the two factor scores

In [11]:
from alpha_research.portfolio import combine_factor_scores

### 3.1 Construct the composite score

In [12]:
composite_panel = panel.copy()

composite_panel["mom_vol_composite_z"] = combine_factor_scores(
    panel=composite_panel,
    factor_weights={
        "mom_12_1m_z": 0.5,
        "realised_vol_63_z": 0.5,
    },
)

composite_targets = build_factor_target_weights(
    panel=composite_panel,
    factor_column="mom_vol_composite_z",
    return_column="forward_ret_1d",
    config=config,
)

composite_daily, composite_holdings = run_target_weight_backtest(
    return_panel=return_panel,
    target_weights=composite_targets,
    return_column="forward_ret_1d",
    transaction_cost_bps=config.transaction_cost_bps,
)

### 3.2 Examine target exposure

In [13]:
composite_target_exposure = (
    composite_targets.assign(
        long_weight=lambda df: df["weight"].clip(lower=0.0),
        short_weight=lambda df: -df["weight"].clip(upper=0.0),
        active_position=lambda df: df["weight"].ne(0.0).astype(int),
    )
    .groupby("date")
    .agg(
        long_exposure=("long_weight", "sum"),
        short_exposure=("short_weight", "sum"),
        active_positions=("active_position", "sum"),
    )
)

composite_target_exposure["gross_exposure"] = (
    composite_target_exposure["long_exposure"]
    + composite_target_exposure["short_exposure"]
)

composite_target_exposure["net_exposure"] = (
    composite_target_exposure["long_exposure"]
    - composite_target_exposure["short_exposure"]
)

composite_target_exposure.agg(
    ["mean", "std", "min", "max"]
).round(4)

,long_exposure,short_exposure,active_positions,gross_exposure,net_exposure
mean,0.9118,0.9118,36.4706,1.8235,0.0
std,0.2839,0.2839,11.3553,0.5678,0.0
min,0.0000,0.0000,0.0000,0.0000,0.0
max,1.0000,1.0000,40.0000,2.0000,0.0


### 3.3 Compare portfolios

In [14]:
comparison_daily = {
    "Momentum": audit_results["12-1 Momentum"]["drift_daily"],
    "Realised Volatility": audit_results["Realised Volatility"]["drift_daily"],
    "50/50 Independent Sleeves": combined_daily,
    "50/50 Composite Score": composite_daily,
}

comparison_rows = []

for portfolio_name, daily in comparison_daily.items():
    gross = summarise_backtest(
        daily,
        return_column="gross_return",
    ).iloc[0]

    net = summarise_backtest(
        daily,
        return_column="net_return",
    ).iloc[0]

    comparison_rows.append(
        {
            "portfolio": portfolio_name,
            "gross_annualised_return": gross["annualised_return"],
            "net_annualised_return": net["annualised_return"],
            "net_annualised_volatility": net["annualised_volatility"],
            "gross_sharpe": gross["sharpe_ratio"],
            "net_sharpe": net["sharpe_ratio"],
            "max_drawdown": net["max_drawdown"],
            "average_rebalance_turnover": net["average_rebalance_turnover"],
            "total_transaction_cost": net["total_transaction_cost"],
            "average_gross_exposure": daily["gross_exposure"].mean(),
        }
    )

comparison_summary = pd.DataFrame(comparison_rows).set_index("portfolio")

comparison_summary.round(4)

,gross_annualised_return,net_annualised_return,net_annualised_volatility,gross_sharpe,net_sharpe,max_drawdown,average_rebalance_turnover,total_transaction_cost,average_gross_exposure
portfolio,,,,,,,,,
Momentum,0.0348,0.0086,0.2119,0.2683,0.1475,-0.5096,0.5080,0.2936,1.8262
Realised Volatility,0.1781,0.1542,0.2395,0.8042,0.7185,-0.4585,0.4063,0.2348,1.9556
50/50 Independent Sleeves,0.1213,0.0982,0.1616,0.7900,0.6610,-0.2566,0.4130,0.2387,1.4712
50/50 Composite Score,0.1519,0.1245,0.2037,0.7966,0.6785,-0.3063,0.4771,0.2758,1.8242


In [15]:
gross_return_comparison = pd.concat(
    {
        name: daily.set_index("date")["gross_return"]
        for name, daily in comparison_daily.items()
    },
    axis=1,
)

gross_return_comparison.corr().round(4)

,Momentum,Realised Volatility,50/50 Independent Sleeves,50/50 Composite Score
Momentum,1.0000,0.0225,0.6696,0.5510
Realised Volatility,0.0225,1.0000,0.7572,0.7886
50/50 Independent Sleeves,0.6696,0.7572,1.0000,0.9457
50/50 Composite Score,0.5510,0.7886,0.9457,1.0000


### 3.4 Findings: 50/50 composite factor score

The composite-score portfolio averages the standardised momentum and realised-volatility scores before ranking stocks and constructing a single long-short portfolio.

Unlike the independent-sleeve method, factor disagreement changes stock rankings rather than directly cancelling positions. The portfolio therefore maintains a higher average gross exposure of approximately 1.82.

The composite produces a net annualised return of 12.45%, volatility of 20.37%, and a net Sharpe ratio of 0.679. It modestly exceeds the independent sleeve portfolio's Sharpe ratio, but has higher turnover, transaction costs, and maximum drawdown.

The two combination methods have a return correlation of approximately 0.95, showing that they primarily express the same underlying factor information through different portfolio-construction rules.

## 4. Controlled-exposure comparison

If both combination methods carry exactly the same target gross exposure on every rebalance date, does the composite score still perform better?

In [16]:
from alpha_research.portfolio import (
    build_factor_target_weights,
    combine_factor_scores,
    combine_sleeve_target_weights,
    rescale_target_weights_to_gross,
)

In [17]:
composite_target_gross_schedule = (
    composite_targets
    .groupby("date")["weight"]
    .agg(lambda weights: weights.abs().sum())
    .rename("composite_target_gross")
)

controlled_sleeve_targets = rescale_target_weights_to_gross(
    target_weights=combined_targets,
    target_gross=composite_target_gross_schedule,
)

controlled_sleeve_daily, controlled_sleeve_holdings = (
    run_target_weight_backtest(
        return_panel=return_panel,
        target_weights=controlled_sleeve_targets,
        return_column="forward_ret_1d",
        transaction_cost_bps=config.transaction_cost_bps,
    )
)

### 4.1 Verify exposure control

In [18]:
def calculate_target_gross(
    targets: pd.DataFrame,
) -> pd.Series:
    return (
        targets.groupby("date")["weight"]
        .agg(lambda weights: weights.abs().sum())
    )


target_gross_audit = pd.concat(
    {
        "Original Independent Sleeves": calculate_target_gross(
            combined_targets
        ),
        "Equal-Exposure Independent Sleeves": calculate_target_gross(
            controlled_sleeve_targets
        ),
        "Composite Score": calculate_target_gross(
            composite_targets
        ),
    },
    axis=1,
).fillna(0.0)

target_gross_audit["controlled_minus_composite"] = (
    target_gross_audit["Equal-Exposure Independent Sleeves"]
    - target_gross_audit["Composite Score"]
)

target_gross_audit.agg(
    ["mean", "std", "min", "max"]
).round(6)

,Original Independent Sleeves,Equal-Exposure Independent Sleeves,Composite Score,controlled_minus_composite
mean,1.470415,1.823529,1.823529,0.0
std,0.381037,0.567765,0.567765,0.0
min,0.000000,0.000000,0.000000,-0.0
max,2.000000,2.000000,2.000000,0.0


### 4.2 Compare performance

In [19]:
controlled_comparison_daily = {
    "Original Independent Sleeves": combined_daily,
    "Equal-Exposure Independent Sleeves": controlled_sleeve_daily,
    "Composite Score": composite_daily,
}

controlled_summary_rows = []

for portfolio_name, daily in controlled_comparison_daily.items():
    gross_summary = summarise_backtest(
        daily,
        return_column="gross_return",
    ).iloc[0]

    net_summary = summarise_backtest(
        daily,
        return_column="net_return",
    ).iloc[0]

    controlled_summary_rows.append(
        {
            "portfolio": portfolio_name,
            "gross_annualised_return": gross_summary[
                "annualised_return"
            ],
            "net_annualised_return": net_summary[
                "annualised_return"
            ],
            "net_annualised_volatility": net_summary[
                "annualised_volatility"
            ],
            "gross_sharpe": gross_summary["sharpe_ratio"],
            "net_sharpe": net_summary["sharpe_ratio"],
            "max_drawdown": net_summary["max_drawdown"],
            "average_rebalance_turnover": net_summary[
                "average_rebalance_turnover"
            ],
            "total_transaction_cost": net_summary[
                "total_transaction_cost"
            ],
            "average_daily_gross_exposure": daily[
                "gross_exposure"
            ].mean(),
        }
    )

controlled_summary = (
    pd.DataFrame(controlled_summary_rows)
    .set_index("portfolio")
)

controlled_summary.round(4)

,gross_annualised_return,net_annualised_return,net_annualised_volatility,gross_sharpe,net_sharpe,max_drawdown,average_rebalance_turnover,total_transaction_cost,average_daily_gross_exposure
portfolio,,,,,,,,,
Original Independent Sleeves,0.1213,0.0982,0.1616,0.7900,0.6610,-0.2566,0.4130,0.2387,1.4712
Equal-Exposure Independent Sleeves,0.1553,0.1224,0.1999,0.8228,0.6779,-0.3173,0.5742,0.3319,1.8240
Composite Score,0.1519,0.1245,0.2037,0.7966,0.6785,-0.3063,0.4771,0.2758,1.8242


In [20]:
matched_exposure_delta = (
    controlled_summary.loc["Composite Score"]
    - controlled_summary.loc[
        "Equal-Exposure Independent Sleeves"
    ]
).rename("composite_minus_equal_exposure_sleeves")

matched_exposure_delta.round(4)

gross_annualised_return        -0.0034
net_annualised_return           0.0022
net_annualised_volatility       0.0038
gross_sharpe                   -0.0262
net_sharpe                      0.0005
max_drawdown                    0.0111
average_rebalance_turnover     -0.0971
total_transaction_cost         -0.0561
average_daily_gross_exposure    0.0002
Name: composite_minus_equal_exposure_sleeves, dtype: float64

In [21]:
controlled_gross_returns = pd.concat(
    {
        portfolio_name: daily.set_index("date")["gross_return"]
        for portfolio_name, daily
        in controlled_comparison_daily.items()
    },
    axis=1,
)

controlled_gross_returns.corr().round(4)

,Original Independent Sleeves,Equal-Exposure Independent Sleeves,Composite Score
Original Independent Sleeves,1.0000,0.9858,0.9457
Equal-Exposure Independent Sleeves,0.9858,1.0000,0.9501
Composite Score,0.9457,0.9501,1.0000


### 4.3 Measure the remaining construction difference

In [22]:
target_pair = (
    controlled_sleeve_targets[
        ["date", "ticker", "weight"]
    ]
    .rename(columns={"weight": "sleeve_weight"})
    .merge(
        composite_targets[
            ["date", "ticker", "weight"]
        ].rename(
            columns={"weight": "composite_weight"}
        ),
        on=["date", "ticker"],
        how="outer",
    )
    .fillna(
        {
            "sleeve_weight": 0.0,
            "composite_weight": 0.0,
        }
    )
)

target_structure_rows = []

for date, date_targets in target_pair.groupby("date"):
    sleeve_weight = date_targets["sleeve_weight"]
    composite_weight = date_targets["composite_weight"]

    target_gross = composite_weight.abs().sum()

    long_overlap = np.minimum(
        sleeve_weight.clip(lower=0.0),
        composite_weight.clip(lower=0.0),
    ).sum()

    short_overlap = np.minimum(
        (-sleeve_weight.clip(upper=0.0)),
        (-composite_weight.clip(upper=0.0)),
    ).sum()

    same_side_overlap = long_overlap + short_overlap

    target_structure_rows.append(
        {
            "date": date,
            "target_gross": target_gross,
            "l1_weight_difference": (
                sleeve_weight - composite_weight
            ).abs().sum(),
            "same_side_weight_overlap": same_side_overlap,
            "same_side_overlap_fraction": (
                same_side_overlap / target_gross
                if target_gross > 0
                else np.nan
            ),
        }
    )

target_structure = pd.DataFrame(target_structure_rows)

target_structure[
    [
        "l1_weight_difference",
        "same_side_weight_overlap",
        "same_side_overlap_fraction",
    ]
].agg(["mean", "std", "min", "max"]).round(4)

,l1_weight_difference,same_side_weight_overlap,same_side_overlap_fraction
mean,1.2697,1.1887,0.6518
std,0.4579,0.3877,0.0605
min,0.0000,0.0000,0.3750
max,2.5000,1.6500,0.8250


### 4.4 Findings: controlled gross exposure

To separate portfolio-construction effects from exposure effects, the independent sleeve targets were rescaled on each rebalance date to match the composite portfolio's target gross exposure. The exposure audit confirms an exact match, with average target gross exposure of approximately 1.82 for both portfolios.

At matched exposure, the independent sleeves produce a slightly higher gross annualised return (15.53% versus 15.19%) and gross Sharpe ratio (0.823 versus 0.797). The composite portfolio, however, has lower average rebalance turnover (0.477 versus 0.574) and lower transaction costs.

Consequently, their net performance is effectively identical: net Sharpe ratios are 0.678 for the independent sleeves and 0.679 for the composite. The composite also has a modestly smaller maximum drawdown (-30.63% versus -31.73%).

The two portfolios have a return correlation of approximately 0.95, while their average same-side target-weight overlap is approximately 65%. They therefore express broadly similar factor information but retain meaningful differences in stock selection and weighting.

Overall, neither construction method is decisively superior at matched exposure. The composite score is marginally more implementation-efficient, while the independent sleeves preserve clearer factor-level attribution. The original sleeve portfolio's smoother risk profile primarily results from natural netting and lower realised gross exposure.

## 5. Dynamic risk-balanced sleeve portfolio

We build a dynamic independent-sleeve portfolio with:

$$
\hat{\sigma}_{k,t}
=
\operatorname{Std}\left(r_{k,t-63:t-1}\right),
\qquad
a_{k,t}
\propto
\frac{1}{\hat{\sigma}_{k,t}}.
$$

NOTE: After auditing, the current portfolio should be described as a shrunk inverse-volatility allocation. The weights are transformed to $0.4 (0.5) + 0.6w_i$, systematically shrinking allocations towards equal weight. It reduces allocation dispersion, but changes average risk contributions from exactly 50/50 to approximately 45.4% momentum and 54.6% realised volatility. 

In [23]:
from alpha_research.portfolio import (
    estimate_trailing_sleeve_volatility,
    calculate_inverse_volatility_allocations,
    combine_dynamic_sleeve_target_weights,
)

In [24]:
def extract_active_gross_returns(
    daily: pd.DataFrame,
    exposure_tolerance: float = 1e-12,
) -> pd.Series:
    indexed = (
        daily.copy()
        .assign(date=lambda df: pd.to_datetime(df["date"]))
        .set_index("date")
        .sort_index()
    )

    return indexed["gross_return"].where(
        indexed["gross_exposure"] > exposure_tolerance
    )


momentum_daily = audit_results[
    "12-1 Momentum"
]["drift_daily"]

volatility_daily = audit_results[
    "Realised Volatility"
]["drift_daily"]

sleeve_return_frame = pd.concat(
    {
        "Momentum": extract_active_gross_returns(
            momentum_daily
        ),
        "Realised Volatility": extract_active_gross_returns(
            volatility_daily
        ),
    },
    axis=1,
).sort_index()

trailing_sleeve_volatility = (
    estimate_trailing_sleeve_volatility(
        sleeve_returns=sleeve_return_frame,
        lookback=63,
        min_periods=42,
        periods_per_year=252,
    )
)

daily_risk_allocations = (
    calculate_inverse_volatility_allocations(
        sleeve_volatility=trailing_sleeve_volatility,
        allocation_floor=0.20,
    )
)

### 5.1 Align allocations with rebalance dates

In [25]:
rebalance_dates = pd.DatetimeIndex(
    combined_targets["date"].unique()
).sort_values()

dynamic_sleeve_allocations = (
    daily_risk_allocations
    .reindex(rebalance_dates, method="ffill")
)

if dynamic_sleeve_allocations.isna().any().any():
    raise ValueError(
        "Dynamic allocations are missing rebalance dates."
    )

dynamic_sleeve_allocations.index.name = "date"

dynamic_allocation_summary = (
    dynamic_sleeve_allocations
    .agg(["mean", "std", "min", "max"])
    .T
)

dynamic_allocation_summary[
    "mean_absolute_change"
] = (
    dynamic_sleeve_allocations
    .diff()
    .abs()
    .mean()
)

dynamic_allocation_summary.round(4)

,mean,std,min,max,mean_absolute_change
Momentum,0.5128,0.0474,0.3737,0.6457,0.0056
Realised Volatility,0.4872,0.0474,0.3543,0.6263,0.0056


In [26]:
rebalance_volatility = (
    trailing_sleeve_volatility
    .reindex(rebalance_dates, method="ffill")
)

valid_risk_estimate = (
    rebalance_volatility.notna().all(axis=1)
)

risk_allocation_start = (
    valid_risk_estimate[valid_risk_estimate]
    .index.min()
)

print("First risk-based allocation date:", risk_allocation_start)
print(
    "Risk-based rebalance fraction:",
    round(valid_risk_estimate.mean(), 4),
)

First risk-based allocation date: 2016-03-14 00:00:00
Risk-based rebalance fraction: 0.8962


### 5.2 Construct and backtest the dynamic portfolio

In [27]:
dynamic_sleeve_targets = (
    combine_dynamic_sleeve_target_weights(
        sleeve_targets={
            "Momentum": momentum_targets,
            "Realised Volatility": volatility_targets,
        },
        sleeve_allocations=dynamic_sleeve_allocations,
    )
)

risk_balanced_daily, risk_balanced_holdings = (
    run_target_weight_backtest(
        return_panel=return_panel,
        target_weights=dynamic_sleeve_targets,
        return_column="forward_ret_1d",
        transaction_cost_bps=config.transaction_cost_bps,
    )
)

### 5.3 Examine the inverse-volatility risk proxy a_k * sig_k

In [28]:
allocated_volatility_proxy = (
    dynamic_sleeve_allocations
    * rebalance_volatility
)

proxy_risk_shares = allocated_volatility_proxy.div(
    allocated_volatility_proxy.sum(axis=1),
    axis=0,
)

proxy_risk_share_summary = (
    proxy_risk_shares
    .loc[valid_risk_estimate]
    .agg(["mean", "std", "min", "max"])
    .T
)

proxy_risk_share_summary.round(4)

,mean,std,min,max
Momentum,0.4899,0.0355,0.3868,0.5943
Realised Volatility,0.5101,0.0355,0.4057,0.6132


### 5.4 Calculate covariance-aware ex-ante risk contributions

In [29]:
risk_contribution_rows = []

for date in rebalance_dates:
    history = (
        sleeve_return_frame.loc[sleeve_return_frame.index < date].dropna().tail(63)
    )

    if len(history) < 42:
        continue

    covariance = history.cov() * 252

    weights = dynamic_sleeve_allocations.loc[date, covariance.columns]

    portfolio_variance = float(
        weights.to_numpy() @ covariance.to_numpy() @ weights.to_numpy()
    )

    if portfolio_variance <= 0.0:
        continue

    marginal_variance = covariance.to_numpy() @ weights.to_numpy()

    contribution_shares = weights.to_numpy() * marginal_variance / portfolio_variance

    row = {"date": date}

    for sleeve_name, contribution in zip(
        covariance.columns,
        contribution_shares,
    ):
        row[sleeve_name] = contribution

    risk_contribution_rows.append(row)

risk_contribution_shares = pd.DataFrame(risk_contribution_rows).set_index("date")

risk_contribution_summary = risk_contribution_shares.agg(
    ["mean", "std", "min", "max"]
).T

risk_contribution_summary.round(4)


,mean,std,min,max
Momentum,0.4538,0.1326,0.0246,0.8980
Realised Volatility,0.5462,0.1326,0.1020,0.9754


### 5.5 Compare portfolio performance

In [30]:
risk_balance_comparison_daily = {
    "Momentum": momentum_daily,
    "Realised Volatility": volatility_daily,
    "Fixed 50/50 Sleeves": combined_daily,
    "Dynamic Risk-Balanced Sleeves": risk_balanced_daily,
    "Composite Score": composite_daily,
}

risk_balance_summary_rows = []

for portfolio_name, daily in (
    risk_balance_comparison_daily.items()
):
    gross_summary = summarise_backtest(
        daily,
        return_column="gross_return",
    ).iloc[0]

    net_summary = summarise_backtest(
        daily,
        return_column="net_return",
    ).iloc[0]

    risk_balance_summary_rows.append(
        {
            "portfolio": portfolio_name,
            "gross_annualised_return": gross_summary[
                "annualised_return"
            ],
            "net_annualised_return": net_summary[
                "annualised_return"
            ],
            "net_annualised_volatility": net_summary[
                "annualised_volatility"
            ],
            "gross_sharpe": gross_summary[
                "sharpe_ratio"
            ],
            "net_sharpe": net_summary[
                "sharpe_ratio"
            ],
            "max_drawdown": net_summary[
                "max_drawdown"
            ],
            "average_rebalance_turnover": net_summary[
                "average_rebalance_turnover"
            ],
            "total_transaction_cost": net_summary[
                "total_transaction_cost"
            ],
            "average_daily_gross_exposure": daily[
                "gross_exposure"
            ].mean(),
        }
    )

risk_balance_summary = (
    pd.DataFrame(risk_balance_summary_rows)
    .set_index("portfolio")
)

risk_balance_summary.round(4)

,gross_annualised_return,net_annualised_return,net_annualised_volatility,gross_sharpe,net_sharpe,max_drawdown,average_rebalance_turnover,total_transaction_cost,average_daily_gross_exposure
portfolio,,,,,,,,,
Momentum,0.0348,0.0086,0.2119,0.2683,0.1475,-0.5096,0.5080,0.2936,1.8262
Realised Volatility,0.1781,0.1542,0.2395,0.8042,0.7185,-0.4585,0.4063,0.2348,1.9556
Fixed 50/50 Sleeves,0.1213,0.0982,0.1616,0.7900,0.6610,-0.2566,0.4130,0.2387,1.4712
Dynamic Risk-Balanced Sleeves,0.1227,0.0988,0.1574,0.8143,0.6777,-0.2103,0.4261,0.2463,1.5074
Composite Score,0.1519,0.1245,0.2037,0.7966,0.6785,-0.3063,0.4771,0.2758,1.8242


In [31]:
risk_balance_gross_returns = pd.concat(
    {
        portfolio_name: daily.set_index("date")[
            "gross_return"
        ]
        for portfolio_name, daily
        in risk_balance_comparison_daily.items()
    },
    axis=1,
)

risk_balance_gross_returns.corr().round(4)

,Momentum,Realised Volatility,Fixed 50/50 Sleeves,Dynamic Risk-Balanced Sleeves,Composite Score
Momentum,1.0000,0.0225,0.6696,0.7154,0.5510
Realised Volatility,0.0225,1.0000,0.7572,0.7042,0.7886
Fixed 50/50 Sleeves,0.6696,0.7572,1.0000,0.9905,0.9457
Dynamic Risk-Balanced Sleeves,0.7154,0.7042,0.9905,1.0000,0.9325
Composite Score,0.5510,0.7886,0.9457,0.9325,1.0000


### 5.6 Inspect dynamic target exposure

In [32]:
dynamic_target_exposure = (
    dynamic_sleeve_targets
    .assign(
        long_weight=lambda df: df["weight"].clip(
            lower=0.0
        ),
        short_weight=lambda df: -df["weight"].clip(
            upper=0.0
        ),
    )
    .groupby("date")
    .agg(
        long_exposure=("long_weight", "sum"),
        short_exposure=("short_weight", "sum"),
    )
)

dynamic_target_exposure["gross_exposure"] = (
    dynamic_target_exposure["long_exposure"]
    + dynamic_target_exposure["short_exposure"]
)

dynamic_target_exposure["net_exposure"] = (
    dynamic_target_exposure["long_exposure"]
    - dynamic_target_exposure["short_exposure"]
)

dynamic_target_exposure.agg(
    ["mean", "std", "min", "max"]
).round(4)

,long_exposure,short_exposure,gross_exposure,net_exposure
mean,0.7533,0.7533,1.5065,-0.0
std,0.1871,0.1871,0.3741,0.0
min,0.0000,0.0000,0.0000,-0.0
max,1.0000,1.0000,2.0000,0.0


### 5.7 Findings: dynamic risk-balanced factor sleeves

The dynamic sleeve portfolio replaces the fixed 50/50 allocation with bounded inverse-volatility weights estimated from trailing, one-day-shifted sleeve returns. Risk-based allocations begin on 14 March 2016 and cover approximately 89.6% of rebalance dates, with equal allocations used during the warm-up period.

The resulting allocations remain moderate and stable. Momentum receives an average allocation of 51.3% and realised volatility 48.7%, while the average absolute allocation change is only 0.56% per rebalance. The volatility-based risk proxy is close to balanced, although covariance-aware contributions average 45.4% for momentum and 54.6% for realised volatility. This difference reflects the fact that inverse-volatility allocation does not explicitly incorporate time-varying covariance.

Relative to the fixed 50/50 sleeves, dynamic allocation modestly improves net annualised return from 9.82% to 9.88% and reduces annualised volatility from 16.16% to 15.74%. Net Sharpe increases from 0.661 to 0.678, while maximum drawdown improves materially from -25.66% to -21.03%. These benefits come with only slightly higher turnover and transaction costs.

The fixed and dynamic portfolios have a return correlation of approximately 0.99, showing that risk balancing refines the existing factor combination rather than introducing a distinct return source. Its principal benefit is a smoother risk path and improved drawdown control, rather than a large increase in headline performance.

### 5.8 Subperiod analysis

In [33]:
subperiod_definitions = {
    "2015-2018": (
        pd.Timestamp("2015-01-01"),
        pd.Timestamp("2018-12-31"),
    ),
    "2019-2022": (
        pd.Timestamp("2019-01-01"),
        pd.Timestamp("2022-12-31"),
    ),
    "2023-Present": (
        pd.Timestamp("2023-01-01"),
        None,
    ),
}

subperiod_portfolios = {
    "Fixed 50/50 Sleeves": combined_daily,
    "Dynamic Risk-Balanced Sleeves": risk_balanced_daily,
    "Composite Score": composite_daily,
}

subperiod_portfolios = {
    name: (
        daily.copy()
        .assign(date=lambda df: pd.to_datetime(df["date"]))
        .sort_values("date")
        .reset_index(drop=True)
    )
    for name, daily in subperiod_portfolios.items()
}

exposure_tolerance = 1e-12

active_start_dates = {
    name: daily.loc[
        daily["gross_exposure"] > exposure_tolerance,
        "date",
    ].min()
    for name, daily in subperiod_portfolios.items()
}

if any(pd.isna(date) for date in active_start_dates.values()):
    raise ValueError("At least one portfolio never becomes active.")

common_evaluation_start = max(active_start_dates.values())

print("Individual active starts:")
for name, date in active_start_dates.items():
    print(f"  {name}: {date.date()}")

print(
    "Common evaluation start:",
    common_evaluation_start.date(),
)

Individual active starts:
  Fixed 50/50 Sleeves: 2015-04-08
  Dynamic Risk-Balanced Sleeves: 2015-04-08
  Composite Score: 2016-01-07
Common evaluation start: 2016-01-07


In [34]:
subperiod_summary_rows = []

for period_name, (period_start, period_end) in subperiod_definitions.items():
    evaluation_start = max(
        period_start,
        common_evaluation_start,
    )

    for portfolio_name, daily in subperiod_portfolios.items():
        mask = daily["date"] >= evaluation_start

        if period_end is not None:
            mask &= daily["date"] <= period_end

        period_daily = daily.loc[mask].copy()

        if period_daily.empty:
            continue

        gross_summary = summarise_backtest(
            period_daily,
            return_column="gross_return",
        ).iloc[0]

        net_summary = summarise_backtest(
            period_daily,
            return_column="net_return",
        ).iloc[0]

        subperiod_summary_rows.append(
            {
                "period": period_name,
                "portfolio": portfolio_name,
                "start_date": period_daily["date"].min(),
                "end_date": period_daily["date"].max(),
                "observations": len(period_daily),
                "gross_annualised_return": gross_summary["annualised_return"],
                "net_annualised_return": net_summary["annualised_return"],
                "net_annualised_volatility": net_summary["annualised_volatility"],
                "gross_sharpe": gross_summary["sharpe_ratio"],
                "net_sharpe": net_summary["sharpe_ratio"],
                "max_drawdown": net_summary["max_drawdown"],
                "average_rebalance_turnover": net_summary["average_rebalance_turnover"],
                "total_transaction_cost": net_summary["total_transaction_cost"],
                "average_daily_gross_exposure": period_daily["gross_exposure"].mean(),
            }
        )

subperiod_summary = pd.DataFrame(subperiod_summary_rows).set_index(
    ["period", "portfolio"]
)

subperiod_summary.round(4)

start_date   end_date  \
period       portfolio                                             
2015-2018    Fixed 50/50 Sleeves           2016-01-07 2018-12-31   
             Dynamic Risk-Balanced Sleeves 2016-01-07 2018-12-31   
             Composite Score               2016-01-07 2018-12-31   
2019-2022    Fixed 50/50 Sleeves           2019-01-02 2022-12-30   
             Dynamic Risk-Balanced Sleeves 2019-01-02 2022-12-30   
             Composite Score               2019-01-02 2022-12-30   
2023-Present Fixed 50/50 Sleeves           2023-01-03 2026-07-01   
             Dynamic Risk-Balanced Sleeves 2023-01-03 2026-07-01   
             Composite Score               2023-01-03 2026-07-01   

                                            observations  \
period       portfolio                                     
2015-2018    Fixed 50/50 Sleeves                     751   
             Dynamic Risk-Balanced Sleeves           751   
             Composite Score                         751   
2019-2022    Fixed 50/50 Sleeves                    1008   
             Dynamic Risk-Balanced Sleeves          1008   
             Composite Score                        1008   
2023-Present Fixed 50/50 Sleeves                     876   
             Dynamic Risk-Balanced Sleeves           876   
             Composite Score                         876   

                                            gross_annualised_return  \
period       portfolio                                                
2015-2018    Fixed 50/50 Sleeves                             0.0557   
             Dynamic Risk-Balanced Sleeves                   0.0519   
             Composite Score                                 0.0735   
2019-2022    Fixed 50/50 Sleeves                             0.0345   
             Dynamic Risk-Balanced Sleeves                   0.0533   
             Composite Score                                 0.0431   
2023-Present Fixed 50/50 Sleeves                             0.3190   
             Dynamic Risk-Balanced Sleeves                   0.3011   
             Composite Score                                 0.4293   

                                            net_annualised_return  \
period       portfolio                                              
2015-2018    Fixed 50/50 Sleeves                           0.0323   
             Dynamic Risk-Balanced Sleeves                 0.0281   
             Composite Score                               0.0460   
2019-2022    Fixed 50/50 Sleeves                           0.0126   
             Dynamic Risk-Balanced Sleeves                 0.0299   
             Composite Score                               0.0151   
2023-Present Fixed 50/50 Sleeves                           0.2901   
             Dynamic Risk-Balanced Sleeves                 0.2718   
             Composite Score                               0.3928   

                                            net_annualised_volatility  \
period       portfolio                                                  
2015-2018    Fixed 50/50 Sleeves                               0.1369   
             Dynamic Risk-Balanced Sleeves                     0.1362   
             Composite Score                                   0.1684   
2019-2022    Fixed 50/50 Sleeves                               0.1538   
             Dynamic Risk-Balanced Sleeves                     0.1438   
             Composite Score                                   0.2019   
2023-Present Fixed 50/50 Sleeves                               0.2048   
             Dynamic Risk-Balanced Sleeves                     0.2028   
             Composite Score                                   0.2560   

                                            gross_sharpe  net_sharpe  \
period       portfolio                                                 
2015-2018    Fixed 50/50 Sleeves                  0.4649      0.3010   
             Dynamic Risk-Balanced Sleeves        0.4400      0.2721   

#### 5.8.1 Calculate relative performance

In [35]:
subperiod_delta_metrics = [
    "net_annualised_return",
    "net_annualised_volatility",
    "net_sharpe",
    "max_drawdown",
    "average_rebalance_turnover",
    "total_transaction_cost",
    "average_daily_gross_exposure",
]

dynamic_subperiod_summary = subperiod_summary.xs(
    "Dynamic Risk-Balanced Sleeves",
    level="portfolio",
)[subperiod_delta_metrics]

fixed_subperiod_summary = subperiod_summary.xs(
    "Fixed 50/50 Sleeves",
    level="portfolio",
)[subperiod_delta_metrics]

composite_subperiod_summary = subperiod_summary.xs(
    "Composite Score",
    level="portfolio",
)[subperiod_delta_metrics]

dynamic_minus_fixed = dynamic_subperiod_summary - fixed_subperiod_summary

dynamic_minus_composite = dynamic_subperiod_summary - composite_subperiod_summary

print("Dynamic minus Fixed 50/50:")
display(dynamic_minus_fixed.round(4))

print("Dynamic minus Composite:")
display(dynamic_minus_composite.round(4))

Dynamic minus Fixed 50/50:


,net_annualised_return,net_annualised_volatility,net_sharpe,max_drawdown,average_rebalance_turnover,total_transaction_cost,average_daily_gross_exposure
period,,,,,,,
2015-2018,-0.0042,-0.0008,-0.0288,-0.0003,0.0078,0.0012,0.0235
2019-2022,0.0173,-0.0099,0.1183,0.0540,0.0212,0.0043,0.0634
2023-Present,-0.0183,-0.0019,-0.0595,-0.0002,0.0121,0.0021,0.0261


Dynamic minus Composite:


,net_annualised_return,net_annualised_volatility,net_sharpe,max_drawdown,average_rebalance_turnover,total_transaction_cost,average_daily_gross_exposure
period,,,,,,,
2015-2018,-0.0179,-0.0322,-0.0797,0.0715,-0.0612,-0.0092,-0.3677
2019-2022,0.0147,-0.0580,0.1012,0.1037,-0.0929,-0.0187,-0.4956
2023-Present,-0.1210,-0.0531,-0.1362,0.0555,-0.0615,-0.0108,-0.3764


#### 5.8.2 Summarise allocations by subperiod

In [36]:
allocation_subperiod_rows = []

for period_name, (period_start, period_end) in subperiod_definitions.items():
    evaluation_start = max(
        period_start,
        common_evaluation_start,
    )

    allocation_mask = dynamic_sleeve_allocations.index >= evaluation_start

    if period_end is not None:
        allocation_mask &= dynamic_sleeve_allocations.index <= period_end

    period_allocations = dynamic_sleeve_allocations.loc[allocation_mask]

    if period_allocations.empty:
        continue

    for sleeve_name in period_allocations.columns:
        sleeve_allocation = period_allocations[sleeve_name]

        allocation_subperiod_rows.append(
            {
                "period": period_name,
                "sleeve": sleeve_name,
                "mean_allocation": sleeve_allocation.mean(),
                "allocation_std": sleeve_allocation.std(),
                "minimum_allocation": sleeve_allocation.min(),
                "maximum_allocation": sleeve_allocation.max(),
                "mean_absolute_change": (sleeve_allocation.diff().abs().mean()),
            }
        )

allocation_subperiod_summary = pd.DataFrame(allocation_subperiod_rows).set_index(
    ["period", "sleeve"]
)

allocation_subperiod_summary.round(4)

mean_allocation  allocation_std  \
period       sleeve                                                 
2015-2018    Momentum                      0.4935          0.0333   
             Realised Volatility           0.5065          0.0333   
2019-2022    Momentum                      0.5280          0.0617   
             Realised Volatility           0.4720          0.0617   
2023-Present Momentum                      0.5157          0.0382   
             Realised Volatility           0.4843          0.0382   

                                  minimum_allocation  maximum_allocation  \
period       sleeve                                                        
2015-2018    Momentum                         0.4373              0.5769   
             Realised Volatility              0.4231              0.5627   
2019-2022    Momentum                         0.3737              0.6457   
             Realised Volatility              0.3543              0.6263   
2023-Present Momentum                         0.4569              0.6059   
             Realised Volatility              0.3941              0.5431   

                                  mean_absolute_change  
period       sleeve                                     
2015-2018    Momentum                           0.0058  
             Realised Volatility                0.0058  
2019-2022    Momentum                           0.0068  
             Realised Volatility                0.0068  
2023-Present Momentum                           0.0057  
             Realised Volatility                0.0057

In [37]:
risk_estimate_subperiod_rows = []

for period_name, (period_start, period_end) in subperiod_definitions.items():
    evaluation_start = max(
        period_start,
        common_evaluation_start,
    )

    period_mask = valid_risk_estimate.index >= evaluation_start

    if period_end is not None:
        period_mask &= valid_risk_estimate.index <= period_end

    period_validity = valid_risk_estimate.loc[period_mask]

    if period_validity.empty:
        continue

    risk_estimate_subperiod_rows.append(
        {
            "period": period_name,
            "rebalance_dates": len(period_validity),
            "risk_based_rebalance_fraction": (period_validity.mean()),
        }
    )

risk_estimate_subperiod_summary = pd.DataFrame(risk_estimate_subperiod_rows).set_index(
    "period"
)

risk_estimate_subperiod_summary.round(4)

,rebalance_dates,risk_based_rebalance_fraction
period,,
2015-2018,151,0.9404
2019-2022,201,1.0000
2023-Present,175,1.0000


#### 5.8.3 Examine covariance-aware risk contributions

In [38]:
risk_contribution_subperiod_rows = []

for period_name, (period_start, period_end) in subperiod_definitions.items():
    evaluation_start = max(
        period_start,
        common_evaluation_start,
    )

    contribution_mask = risk_contribution_shares.index >= evaluation_start

    if period_end is not None:
        contribution_mask &= risk_contribution_shares.index <= period_end

    period_contributions = risk_contribution_shares.loc[contribution_mask]

    if period_contributions.empty:
        continue

    for sleeve_name in period_contributions.columns:
        sleeve_contribution = period_contributions[sleeve_name]

        risk_contribution_subperiod_rows.append(
            {
                "period": period_name,
                "sleeve": sleeve_name,
                "mean_risk_contribution": (sleeve_contribution.mean()),
                "risk_contribution_std": (sleeve_contribution.std()),
                "minimum_risk_contribution": (sleeve_contribution.min()),
                "maximum_risk_contribution": (sleeve_contribution.max()),
            }
        )

risk_contribution_subperiod_summary = pd.DataFrame(
    risk_contribution_subperiod_rows
).set_index(["period", "sleeve"])

risk_contribution_subperiod_summary.round(4)


mean_risk_contribution  \
period       sleeve                                        
2015-2018    Momentum                             0.4799   
             Realised Volatility                  0.5201   
2019-2022    Momentum                             0.4117   
             Realised Volatility                  0.5883   
2023-Present Momentum                             0.4809   
             Realised Volatility                  0.5191   

                                  risk_contribution_std  \
period       sleeve                                       
2015-2018    Momentum                            0.0890   
             Realised Volatility                 0.0890   
2019-2022    Momentum                            0.1738   
             Realised Volatility                 0.1738   
2023-Present Momentum                            0.0882   
             Realised Volatility                 0.0882   

                                  minimum_risk_contribution  \
period       sleeve                                           
2015-2018    Momentum                                0.1385   
             Realised Volatility                     0.4434   
2019-2022    Momentum                                0.0246   
             Realised Volatility                     0.2517   
2023-Present Momentum                                0.2373   
             Realised Volatility                     0.1020   

                                  maximum_risk_contribution  
period       sleeve                                          
2015-2018    Momentum                                0.5566  
             Realised Volatility                     0.8615  
2019-2022    Momentum                                0.7483  
             Realised Volatility                     0.9754  
2023-Present Momentum                                0.8980  
             Realised Volatility                     0.7627

#### 5.8.4 Subperiod findings

The three main portfolio constructions were compared over 2016–2018, 2019–2022, and 2023–present using a common evaluation start of 7 January 2016. The first period therefore excludes 2015 because the composite signal was not yet active.

Dynamic risk balancing does not consistently outperform the fixed 50/50 allocation. Its clearest benefit occurs during 2019–2022, when it improves net annualised return by 1.73 percentage points, reduces volatility by 0.99 percentage points, raises net Sharpe by 0.118, and reduces maximum drawdown from -25.66% to -20.26%. In 2016–2018 and 2023–present, however, the fixed allocation produces slightly higher return and Sharpe, with almost identical drawdowns.

Consequently, the dynamic portfolio's modest full-sample improvement is largely attributable to the 2019–2022 period rather than a persistent advantage across regimes. It should therefore be interpreted as a defensive risk-management refinement, not a reliable factor-timing mechanism.

Relative to the composite score, the dynamic sleeves consistently carry lower volatility, gross exposure, turnover, transaction costs, and drawdown. The composite nevertheless produces higher return and Sharpe in 2016–2018 and 2023–present. Part of this return difference reflects its materially higher gross exposure, averaging approximately 2.00 versus 1.51–1.63 for the dynamic portfolio.

Dynamic allocations remain moderate across all periods, but covariance-aware risk contributions are not consistently balanced. The imbalance is greatest during 2019–2022, despite the method's strongest performance improvement. The current approach therefore balances estimated standalone sleeve volatility more closely than total portfolio risk.

Overall, fixed 50/50 sleeves remain the simplest and most robust baseline. Dynamic risk balancing is retained as a useful alternative with better full-sample downside behaviour, while the composite remains the more aggressive, implementation-efficient construction.

### 5.9 Risk-allocation implementation audit

In [39]:
import math

In [40]:
def impose_proportional_allocation_floor(
    weights: pd.Series,
    allocation_floor: float,
) -> pd.Series:
    """Impose a true lower bound while preserving relative free weights."""
    weights = weights.astype(float)

    if (
        weights.isna().any()
        or not np.isfinite(weights.to_numpy()).all()
        or (weights < 0.0).any()
    ):
        raise ValueError("weights must be finite and non-negative.")

    if not np.isclose(weights.sum(), 1.0):
        raise ValueError("weights must sum to one.")

    asset_count = len(weights)

    if allocation_floor < 0.0 or allocation_floor * asset_count > 1.0:
        raise ValueError("Invalid allocation floor.")

    result = pd.Series(
        0.0,
        index=weights.index,
        dtype=float,
    )

    remaining_assets = list(weights.index)
    remaining_capital = 1.0

    while remaining_assets:
        remaining_weights = weights.loc[remaining_assets]

        candidate = remaining_weights / remaining_weights.sum() * remaining_capital

        floor_violations = candidate[candidate < allocation_floor].index.tolist()

        if not floor_violations:
            result.loc[remaining_assets] = candidate
            break

        result.loc[floor_violations] = allocation_floor

        remaining_capital -= allocation_floor * len(floor_violations)

        remaining_assets = [
            name for name in remaining_assets if name not in floor_violations
        ]

    return result


def calculate_component_risk_shares(
    weights: pd.Series,
    covariance: pd.DataFrame,
) -> tuple[pd.Series, float]:
    """Calculate covariance-aware component risk shares."""
    covariance = covariance.loc[
        weights.index,
        weights.index,
    ]

    weight_array = weights.to_numpy()
    covariance_array = covariance.to_numpy()

    portfolio_variance = float(weight_array @ covariance_array @ weight_array)

    if portfolio_variance <= 0.0:
        raise ValueError("Portfolio variance must be positive.")

    component_variance = weight_array * (covariance_array @ weight_array)

    risk_shares = pd.Series(
        component_variance / portfolio_variance,
        index=weights.index,
    )

    return risk_shares, math.sqrt(portfolio_variance)

#### 5.9.1 Reconstruct all methods from aligned inputs

In [41]:
allocation_floor = 0.20
lookback = 63
min_periods = 42
periods_per_year = 252

sleeve_names = list(dynamic_sleeve_allocations.columns)

if set(sleeve_names) != set(sleeve_return_frame.columns):
    raise ValueError("Sleeve names do not match return columns.")

allocation_audit_rows = []
covariance_diagnostic_rows = []

for date in rebalance_dates:
    history = (
        sleeve_return_frame.loc[
            sleeve_return_frame.index < date,
            sleeve_names,
        ]
        .dropna()
        .tail(lookback)
    )

    if len(history) < min_periods:
        continue

    covariance = history.cov() * periods_per_year

    volatility = pd.Series(
        np.sqrt(np.diag(covariance)),
        index=sleeve_names,
    )

    if (
        volatility.isna().any()
        or not np.isfinite(volatility.to_numpy()).all()
        or (volatility <= 0.0).any()
    ):
        continue

    inverse_volatility = 1.0 / volatility

    pure_weights = inverse_volatility / inverse_volatility.sum()

    true_floor_weights = impose_proportional_allocation_floor(
        weights=pure_weights,
        allocation_floor=allocation_floor,
    )

    current_shrinkage_weights = (
        allocation_floor + (1.0 - len(sleeve_names) * allocation_floor) * pure_weights
    )

    existing_dynamic_weights = dynamic_sleeve_allocations.loc[
        date, sleeve_names
    ].astype(float)

    pairwise_correlation = covariance.iloc[0, 1] / (
        volatility.iloc[0] * volatility.iloc[1]
    )

    covariance_diagnostic_rows.append(
        {
            "date": date,
            "observations": len(history),
            "sleeve_correlation": pairwise_correlation,
            "pure_minimum_allocation": (pure_weights.min()),
            "floor_binds": bool((pure_weights < allocation_floor).any()),
        }
    )

    allocation_methods = {
        "Pure inverse volatility": pure_weights,
        "True 20% floor": true_floor_weights,
        "Aligned current shrinkage": (current_shrinkage_weights),
        "Existing dynamic weights": (existing_dynamic_weights),
    }

    for method_name, weights in allocation_methods.items():
        risk_shares, portfolio_volatility = calculate_component_risk_shares(
            weights=weights,
            covariance=covariance,
        )

        row = {
            "date": date,
            "method": method_name,
            "sleeve_correlation": pairwise_correlation,
            "portfolio_volatility": (portfolio_volatility),
        }

        for sleeve_name in sleeve_names:
            row[f"{sleeve_name}_allocation"] = weights[sleeve_name]

            row[f"{sleeve_name}_allocated_volatility"] = (
                weights[sleeve_name] * volatility[sleeve_name]
            )

            row[f"{sleeve_name}_risk_contribution"] = risk_shares[sleeve_name]

        allocation_audit_rows.append(row)

allocation_audit = pd.DataFrame(allocation_audit_rows)

covariance_diagnostics = pd.DataFrame(covariance_diagnostic_rows).set_index("date")

print(
    "Audited rebalance dates:",
    covariance_diagnostics.shape[0],
)
print(
    "True floor binding fraction:",
    round(
        covariance_diagnostics["floor_binds"].mean(),
        4,
    ),
)
print(
    "Minimum pure allocation:",
    round(
        covariance_diagnostics["pure_minimum_allocation"].min(),
        4,
    ),
)

covariance_diagnostics["sleeve_correlation"].describe().round(4)

Audited rebalance dates: 518
True floor binding fraction: 0.0
Minimum pure allocation: 0.2571


count    518.0000
mean       0.1613
std        0.6141
min       -0.9800
25%       -0.4449
50%        0.4137
75%        0.7069
max        0.8913
Name: sleeve_correlation, dtype: float64

#### 5.9.2 Compare allocation and risk-contribution behaviour

In [42]:
allocation_risk_summary_rows = []

for method_name, method_frame in allocation_audit.groupby("method"):
    for sleeve_name in sleeve_names:
        allocation = method_frame[f"{sleeve_name}_allocation"]

        risk_contribution = method_frame[f"{sleeve_name}_risk_contribution"]

        allocation_risk_summary_rows.append(
            {
                "method": method_name,
                "sleeve": sleeve_name,
                "mean_allocation": allocation.mean(),
                "allocation_std": allocation.std(),
                "minimum_allocation": allocation.min(),
                "maximum_allocation": allocation.max(),
                "mean_risk_contribution": (risk_contribution.mean()),
                "risk_contribution_std": (risk_contribution.std()),
                "minimum_risk_contribution": (risk_contribution.min()),
                "maximum_risk_contribution": (risk_contribution.max()),
                "mean_absolute_deviation_from_50pct": (
                    risk_contribution.sub(0.5).abs().mean()
                ),
                "maximum_absolute_deviation_from_50pct": (
                    risk_contribution.sub(0.5).abs().max()
                ),
            }
        )

allocation_risk_summary = pd.DataFrame(allocation_risk_summary_rows).set_index(
    ["method", "sleeve"]
)

allocation_risk_summary.round(4)

mean_allocation  \
method                    sleeve                                 
Aligned current shrinkage Momentum                      0.5143   
                          Realised Volatility           0.4857   
Existing dynamic weights  Momentum                      0.5142   
                          Realised Volatility           0.4858   
Pure inverse volatility   Momentum                      0.5239   
                          Realised Volatility           0.4761   
True 20% floor            Momentum                      0.5239   
                          Realised Volatility           0.4761   

                                               allocation_std  \
method                    sleeve                                
Aligned current shrinkage Momentum                     0.0498   
                          Realised Volatility          0.0498   
Existing dynamic weights  Momentum                     0.0498   
                          Realised Volatility          0.0498   
Pure inverse volatility   Momentum                     0.0831   
                          Realised Volatility          0.0831   
True 20% floor            Momentum                     0.0831   
                          Realised Volatility          0.0831   

                                               minimum_allocation  \
method                    sleeve                                    
Aligned current shrinkage Momentum                         0.3737   
                          Realised Volatility              0.3543   
Existing dynamic weights  Momentum                         0.3737   
                          Realised Volatility              0.3543   
Pure inverse volatility   Momentum                         0.2894   
                          Realised Volatility              0.2571   
True 20% floor            Momentum                         0.2894   
                          Realised Volatility              0.2571   

                                               maximum_allocation  \
method                    sleeve                                    
Aligned current shrinkage Momentum                         0.6457   
                          Realised Volatility              0.6263   
Existing dynamic weights  Momentum                         0.6457   
                          Realised Volatility              0.6263   
Pure inverse volatility   Momentum                         0.7429   
                          Realised Volatility              0.7106   
True 20% floor            Momentum                         0.7429   
                          Realised Volatility              0.7106   

                                               mean_risk_contribution  \
method                    sleeve                                        
Aligned current shrinkage Momentum                             0.4539   
                          Realised Volatility                  0.5461   
Existing dynamic weights  Momentum                             0.4538   
                          Realised Volatility                  0.5462   
Pure inverse volatility   Momentum                             0.5000   
                          Realised Volatility                  0.5000   
True 20% floor            Momentum                             0.5000   
                          Realised Volatility                  0.5000   

                                               risk_contribution_std  \
method                    sleeve                                       
Aligned current shrinkage Momentum                            0.1326   
                          Realised Volatility                 0.1326   
Existing dynamic weights  Momentum                            0.1326   
                          Realised Volatility                 0.1326   
Pure inverse volatility   Momentum                            0.0000   
                          Realised Volatility                 0.0000   
True 20% floor            Momentum            

#### 5.9.3 Verify the inverse-volatility identity

In [43]:
pure_audit = allocation_audit.loc[
    allocation_audit["method"] == "Pure inverse volatility"
].set_index("date")

allocated_volatility_difference = (
    pure_audit[f"{sleeve_names[0]}_allocated_volatility"]
    - pure_audit[f"{sleeve_names[1]}_allocated_volatility"]
)

pure_risk_deviation = pure_audit[f"{sleeve_names[0]}_risk_contribution"] - 0.5

pure_inverse_volatility_check = pd.Series(
    {
        "maximum_absolute_allocated_volatility_difference": (
            allocated_volatility_difference.abs().max()
        ),
        "mean_absolute_risk_contribution_deviation": (pure_risk_deviation.abs().mean()),
        "maximum_absolute_risk_contribution_deviation": (
            pure_risk_deviation.abs().max()
        ),
    }
)

pure_inverse_volatility_check

maximum_absolute_allocated_volatility_difference    2.775558e-17
mean_absolute_risk_contribution_deviation           1.293474e-16
maximum_absolute_risk_contribution_deviation        5.107026e-15
dtype: float64

#### 5.9.4 Quantify the estimation-alignment difference

In [44]:
def extract_method_allocations(
    audit: pd.DataFrame,
    method_name: str,
    sleeve_names: list[str],
) -> pd.DataFrame:
    allocation_columns = {f"{name}_allocation": name for name in sleeve_names}

    return (
        audit.loc[
            audit["method"] == method_name,
            ["date", *allocation_columns],
        ]
        .set_index("date")
        .rename(columns=allocation_columns)
        .sort_index()
    )


aligned_shrinkage_allocations = extract_method_allocations(
    audit=allocation_audit,
    method_name="Aligned current shrinkage",
    sleeve_names=sleeve_names,
)

existing_audited_allocations = extract_method_allocations(
    audit=allocation_audit,
    method_name="Existing dynamic weights",
    sleeve_names=sleeve_names,
)

allocation_alignment_difference = (
    existing_audited_allocations - aligned_shrinkage_allocations
)

allocation_alignment_summary = pd.DataFrame(
    {
        "mean_signed_difference": (allocation_alignment_difference.mean()),
        "mean_absolute_difference": (allocation_alignment_difference.abs().mean()),
        "maximum_absolute_difference": (allocation_alignment_difference.abs().max()),
    }
)

allocation_alignment_summary.round(6)

,mean_signed_difference,mean_absolute_difference,maximum_absolute_difference
Momentum,-0.000088,0.000088,0.021008
Realised Volatility,0.000088,0.000088,0.021008


#### 5.9.5 Inspect the most imbalanced dates

In [45]:
risk_contribution_columns = [f"{name}_risk_contribution" for name in sleeve_names]

allocation_audit["maximum_absolute_risk_deviation"] = (
    allocation_audit[risk_contribution_columns].sub(0.5).abs().max(axis=1)
)

worst_risk_balance_dates = (
    allocation_audit.sort_values(
        [
            "method",
            "maximum_absolute_risk_deviation",
        ],
        ascending=[True, False],
    )
    .groupby("method", as_index=False)
    .head(5)
)

display_columns = [
    "date",
    "method",
    "sleeve_correlation",
    "portfolio_volatility",
    "maximum_absolute_risk_deviation",
]

for sleeve_name in sleeve_names:
    display_columns.extend(
        [
            f"{sleeve_name}_allocation",
            f"{sleeve_name}_risk_contribution",
        ]
    )

worst_risk_balance_dates[display_columns].round(4)

,date,method,sleeve_correlation,portfolio_volatility,maximum_absolute_risk_deviation,Momentum_allocation,Momentum_risk_contribution,Realised Volatility_allocation,Realised Volatility_risk_contribution
1294,2022-08-11,Aligned current shrinkage,-0.8503,0.0929,0.4754,0.5560,0.0246,0.4440,0.9754
1262,2022-06-14,Aligned current shrinkage,-0.6761,0.1245,0.4729,0.6208,0.0271,0.3792,0.9729
1298,2022-08-18,Aligned current shrinkage,-0.8674,0.0845,0.4503,0.5466,0.0497,0.4534,0.9503
1250,2022-05-23,Aligned current shrinkage,-0.5925,0.1336,0.4475,0.6404,0.0525,0.3596,0.9475
1258,2022-06-07,Aligned current shrinkage,-0.6216,0.1303,0.4473,0.6312,0.0527,0.3688,0.9473
1295,2022-08-11,Existing dynamic weights,-0.8503,0.0929,0.4754,0.5560,0.0246,0.4440,0.9754
1263,2022-06-14,Existing dynamic weights,-0.6761,0.1245,0.4729,0.6208,0.0271,0.3792,0.9729
1299,2022-08-18,Existing dynamic weights,-0.8674,0.0845,0.4503,0.5466,0.0497,0.4534,0.9503
1251,2022-05-23,Existing dynamic weights,-0.5925,0.1336,0.4475,0.6404,0.0525,0.3596,0.9475
1259,2022-06-07,Existing dynamic weights,-0.6216,0.1303,0.4473,0.6312,0.0527,0.3688,0.9473


#### 5.9.6 Implementation-audit findings

The dynamic allocation procedure was audited using identical trailing return windows for volatility, covariance, allocations, and ex-ante component risk contributions.

Pure inverse-volatility allocation produces exactly equal component risk contributions for the two sleeves, up to floating-point error. The maximum absolute deviation from a 50% contribution is approximately zero. This confirms the theoretical result that, for two positive-weight sleeves, equal allocated standalone volatility implies equal component risk contributions regardless of their correlation.

The genuine 20% allocation floor never binds. The minimum unconstrained allocation is 25.71%, so true floor-constrained weights are identical to pure inverse-volatility weights throughout the audited sample. 

The previously implemented rule instead transforms each inverse-volatility weight according to $0.20 + 0.60w_i$, systematically shrinking allocations towards equal weight. It reduces allocation dispersion, but changes average risk contributions from exactly 50/50 to approximately 45.4% momentum and 54.6% realised volatility. On strongly negatively correlated dates, the resulting risk-contribution imbalance becomes extreme because portfolio variance is small and modest allocated-volatility differences are amplified.

Differences caused by estimation-window alignment are negligible on average. The systematic imbalance is therefore attributable primarily to the shrinkage rule rather than inconsistent volatility and covariance inputs.

The current portfolio should consequently be described as a shrunk inverse-volatility allocation, not a floor-constrained risk-balanced portfolio. A genuine inverse-volatility version should be tested separately before selecting the preferred production construction.

## 6. Pure risk-balanced sleeve portfolio

In [46]:
pure_audited_allocations = extract_method_allocations(
    audit=allocation_audit,
    method_name="Pure inverse volatility",
    sleeve_names=sleeve_names,
)

pure_inverse_volatility_allocations = pd.DataFrame(
    1.0 / len(sleeve_names),
    index=pd.DatetimeIndex(rebalance_dates).sort_values().unique(),
    columns=sleeve_names,
)

pure_inverse_volatility_allocations.index.name = "date"

pure_inverse_volatility_allocations.update(pure_audited_allocations)

if not np.allclose(
    pure_inverse_volatility_allocations.sum(axis=1),
    1.0,
):
    raise ValueError("Sleeve allocations do not sum to one.")

if (pure_inverse_volatility_allocations < 0.0).any().any():
    raise ValueError("Sleeve allocations must be non-negative.")

pure_risk_based_dates = pure_inverse_volatility_allocations.index.isin(
    pure_audited_allocations.index
)

print(
    "Rebalance dates:",
    len(pure_inverse_volatility_allocations),
)
print(
    "Risk-based dates:",
    pure_risk_based_dates.sum(),
)
print(
    "Risk-based fraction:",
    round(pure_risk_based_dates.mean(), 4),
)
print(
    "First risk-based date:",
    pure_audited_allocations.index.min().date(),
)

pure_inverse_volatility_allocations.describe().round(4)

Rebalance dates: 578
Risk-based dates: 518
Risk-based fraction: 0.8962
First risk-based date: 2016-03-14


,Momentum,Realised Volatility
count,578.0000,578.0000
mean,0.5214,0.4786
std,0.0790,0.0790
min,0.2894,0.2571
25%,0.4756,0.4292
50%,0.5072,0.4928
75%,0.5708,0.5244
max,0.7429,0.7106


### 6.1 Construct and backtest the stock-level portfolio

In [47]:
pure_inverse_volatility_targets = (
    combine_dynamic_sleeve_target_weights(
        sleeve_targets={
            "Momentum": momentum_targets,
            "Realised Volatility": volatility_targets,
        },
        sleeve_allocations=(
            pure_inverse_volatility_allocations
        ),
    )
)

(
    pure_inverse_volatility_daily,
    pure_inverse_volatility_holdings,
) = run_target_weight_backtest(
    return_panel=return_panel,
    target_weights=pure_inverse_volatility_targets,
    return_column="forward_ret_1d",
    transaction_cost_bps=config.transaction_cost_bps,
)

print(
    "Target rows:",
    len(pure_inverse_volatility_targets),
)
print(
    "Daily observations:",
    len(pure_inverse_volatility_daily),
)
print(
    "Active start:",
    pure_inverse_volatility_daily.loc[
        pure_inverse_volatility_daily[
            "gross_exposure"
        ] > 1e-12,
        "date",
    ].min().date(),
)

Target rows: 56828
Daily observations: 2890
Active start: 2015-04-08


### 6.2 Compare all four portfolios on identical dates

In [48]:
four_portfolios = {
    "Fixed 50/50 Sleeves": combined_daily,
    "Shrunk Inverse Volatility": risk_balanced_daily,
    "Pure Inverse Volatility": (
        pure_inverse_volatility_daily
    ),
    "Composite Score": composite_daily,
}

four_portfolios = {
    name: (
        daily.copy()
        .assign(date=lambda df: pd.to_datetime(df["date"]))
        .sort_values("date")
        .reset_index(drop=True)
    )
    for name, daily in four_portfolios.items()
}

active_start_dates = {
    name: daily.loc[
        daily["gross_exposure"] > 1e-12,
        "date",
    ].min()
    for name, daily in four_portfolios.items()
}

common_four_portfolio_start = max(
    active_start_dates.values()
)

print("Active starts:")

for name, date in active_start_dates.items():
    print(f"  {name}: {date.date()}")

print(
    "Common evaluation start:",
    common_four_portfolio_start.date(),
)

Active starts:
  Fixed 50/50 Sleeves: 2015-04-08
  Shrunk Inverse Volatility: 2015-04-08
  Pure Inverse Volatility: 2015-04-08
  Composite Score: 2016-01-07
Common evaluation start: 2016-01-07


In [49]:
four_portfolio_summary_rows = []

for portfolio_name, daily in four_portfolios.items():
    evaluation_daily = daily.loc[daily["date"] >= common_four_portfolio_start].copy()

    gross_summary = summarise_backtest(
        evaluation_daily,
        return_column="gross_return",
    ).iloc[0]

    net_summary = summarise_backtest(
        evaluation_daily,
        return_column="net_return",
    ).iloc[0]

    four_portfolio_summary_rows.append(
        {
            "portfolio": portfolio_name,
            "start_date": evaluation_daily["date"].min(),
            "end_date": evaluation_daily["date"].max(),
            "observations": len(evaluation_daily),
            "gross_annualised_return": gross_summary["annualised_return"],
            "net_annualised_return": net_summary["annualised_return"],
            "net_annualised_volatility": net_summary["annualised_volatility"],
            "gross_sharpe": gross_summary["sharpe_ratio"],
            "net_sharpe": net_summary["sharpe_ratio"],
            "max_drawdown": net_summary["max_drawdown"],
            "average_rebalance_turnover": net_summary["average_rebalance_turnover"],
            "total_transaction_cost": net_summary["total_transaction_cost"],
            "average_daily_gross_exposure": (evaluation_daily["gross_exposure"].mean()),
        }
    )

four_portfolio_summary = pd.DataFrame(four_portfolio_summary_rows).set_index(
    "portfolio"
)

four_portfolio_summary.round(4)

,start_date,end_date,observations,gross_annualised_return,net_annualised_return,net_annualised_volatility,gross_sharpe,net_sharpe,max_drawdown,average_rebalance_turnover,total_transaction_cost,average_daily_gross_exposure
portfolio,,,,,,,,,,,,
Fixed 50/50 Sleeves,2016-01-07,2026-07-01,2635,0.1280,0.1035,0.1684,0.8000,0.6695,-0.2566,0.4355,0.2295,1.5415
Shrunk Inverse Volatility,2016-01-07,2026-07-01,2635,0.1295,0.1042,0.1640,0.8250,0.6866,-0.2103,0.4499,0.2371,1.5812
Pure Inverse Volatility,2016-01-07,2026-07-01,2635,0.1301,0.1041,0.1629,0.8330,0.6899,-0.1957,0.4619,0.2434,1.6079
Composite Score,2016-01-07,2026-07-01,2635,0.1678,0.1374,0.2133,0.8343,0.7106,-0.3063,0.5233,0.2758,2.0007


In [50]:
comparison_metrics = [
    "net_annualised_return",
    "net_annualised_volatility",
    "net_sharpe",
    "max_drawdown",
    "average_rebalance_turnover",
    "total_transaction_cost",
    "average_daily_gross_exposure",
]

pure_summary = four_portfolio_summary.loc[
    "Pure Inverse Volatility",
    comparison_metrics,
]

pure_minus_benchmarks = pd.DataFrame(
    {
        benchmark_name: (
            pure_summary
            - four_portfolio_summary.loc[
                benchmark_name,
                comparison_metrics,
            ]
        )
        for benchmark_name in [
            "Fixed 50/50 Sleeves",
            "Shrunk Inverse Volatility",
            "Composite Score",
        ]
    }
).T

pure_minus_benchmarks.index.name = "Pure inverse volatility minus"

pure_minus_benchmarks.round(4)

,net_annualised_return,net_annualised_volatility,net_sharpe,max_drawdown,average_rebalance_turnover,total_transaction_cost,average_daily_gross_exposure
Pure inverse volatility minus,,,,,,,
Fixed 50/50 Sleeves,0.0006,-0.0055,0.0203,0.0609,0.0264,0.0139,0.0663
Shrunk Inverse Volatility,-0.0001,-0.0011,0.0033,0.0147,0.0121,0.0064,0.0267
Composite Score,-0.0333,-0.0504,-0.0207,0.1106,-0.0613,-0.0323,-0.3929


### 6.3 Allocation stability

In [51]:
allocation_comparison_rows = []

allocation_methods = {
    "Pure Inverse Volatility": (
        pure_inverse_volatility_allocations
    ),
    "Shrunk Inverse Volatility": (
        dynamic_sleeve_allocations
    ),
}

for method_name, allocations in (
    allocation_methods.items()
):
    allocations = allocations.loc[
        allocations.index >= common_four_portfolio_start
    ]

    for sleeve_name in sleeve_names:
        sleeve_allocation = allocations[sleeve_name]

        allocation_comparison_rows.append(
            {
                "method": method_name,
                "sleeve": sleeve_name,
                "mean_allocation": sleeve_allocation.mean(),
                "allocation_std": sleeve_allocation.std(),
                "minimum_allocation": sleeve_allocation.min(),
                "maximum_allocation": sleeve_allocation.max(),
                "mean_absolute_change": (
                    sleeve_allocation.diff().abs().mean()
                ),
            }
        )

pure_allocation_comparison = (
    pd.DataFrame(allocation_comparison_rows)
    .set_index(["method", "sleeve"])
)

pure_allocation_comparison.round(4)

mean_allocation  \
method                    sleeve                                 
Pure Inverse Volatility   Momentum                      0.5235   
                          Realised Volatility           0.4765   
Shrunk Inverse Volatility Momentum                      0.5140   
                          Realised Volatility           0.4860   

                                               allocation_std  \
method                    sleeve                                
Pure Inverse Volatility   Momentum                     0.0824   
                          Realised Volatility          0.0824   
Shrunk Inverse Volatility Momentum                     0.0494   
                          Realised Volatility          0.0494   

                                               minimum_allocation  \
method                    sleeve                                    
Pure Inverse Volatility   Momentum                         0.2894   
                          Realised Volatility              0.2571   
Shrunk Inverse Volatility Momentum                         0.3737   
                          Realised Volatility              0.3543   

                                               maximum_allocation  \
method                    sleeve                                    
Pure Inverse Volatility   Momentum                         0.7429   
                          Realised Volatility              0.7106   
Shrunk Inverse Volatility Momentum                         0.6457   
                          Realised Volatility              0.6263   

                                               mean_absolute_change  
method                    sleeve                                     
Pure Inverse Volatility   Momentum                           0.0103  
                          Realised Volatility                0.0103  
Shrunk Inverse Volatility Momentum                           0.0061  
                          Realised Volatility                0.0061

### 6.4 Findings: pure inverse-volatility portfolio

A genuine inverse-volatility sleeve portfolio was constructed using lagged 63-day volatility estimates. Because the minimum unconstrained sleeve allocation is 25.71%, the nominal 20% allocation floor never binds.

Relative to the fixed 50/50 portfolio, pure inverse-volatility allocation reduces annualised volatility from 16.84% to 16.29%, increases net Sharpe from 0.670 to 0.690, and improves maximum drawdown from -25.66% to -19.57%. Net annualised return remains almost unchanged at approximately 10.4%.

Relative to the shrunk inverse-volatility variant, however, the improvement is small. Pure inverse volatility produces nearly identical net return and only a 0.003 increase in net Sharpe. Its maximum drawdown is 1.47 percentage points smaller, but it incurs moderately higher turnover and transaction costs.

Pure inverse-volatility allocations are more responsive than the shrunk allocations. Their standard deviation is approximately 8.2%, compared with 4.9% under shrinkage, and their mean absolute rebalance change is 1.03% rather than 0.61%. Nevertheless, allocations remain within a reasonable range of approximately 26% to 74%.

Overall, pure inverse volatility provides the theoretically correct risk-balanced construction and improves downside behaviour, but its full-sample advantage over the shrunk variant is too small to establish clear superiority. Subperiod analysis is required to determine whether the drawdown benefit is persistent or concentrated in a particular regime.

### 6.5 Subperiod analysis

In [52]:
pure_subperiod_rows = []

for period_name, (period_start, period_end) in subperiod_definitions.items():
    evaluation_start = max(
        period_start,
        common_four_portfolio_start,
    )

    for portfolio_name, daily in four_portfolios.items():
        period_mask = daily["date"] >= evaluation_start

        if period_end is not None:
            period_mask &= daily["date"] <= period_end

        period_daily = daily.loc[period_mask].copy()

        if period_daily.empty:
            continue

        net_summary = summarise_backtest(
            period_daily,
            return_column="net_return",
        ).iloc[0]

        pure_subperiod_rows.append(
            {
                "period": period_name,
                "portfolio": portfolio_name,
                "start_date": period_daily["date"].min(),
                "end_date": period_daily["date"].max(),
                "observations": len(period_daily),
                "net_annualised_return": net_summary["annualised_return"],
                "net_annualised_volatility": net_summary["annualised_volatility"],
                "net_sharpe": net_summary["sharpe_ratio"],
                "max_drawdown": net_summary["max_drawdown"],
                "average_rebalance_turnover": net_summary["average_rebalance_turnover"],
                "total_transaction_cost": net_summary["total_transaction_cost"],
                "average_daily_gross_exposure": (period_daily["gross_exposure"].mean()),
            }
        )

pure_subperiod_summary = pd.DataFrame(pure_subperiod_rows).set_index(
    ["period", "portfolio"]
)

pure_subperiod_summary.round(4)

start_date   end_date  observations  \
period       portfolio                                                       
2015-2018    Fixed 50/50 Sleeves       2016-01-07 2018-12-31           751   
             Shrunk Inverse Volatility 2016-01-07 2018-12-31           751   
             Pure Inverse Volatility   2016-01-07 2018-12-31           751   
             Composite Score           2016-01-07 2018-12-31           751   
2019-2022    Fixed 50/50 Sleeves       2019-01-02 2022-12-30          1008   
             Shrunk Inverse Volatility 2019-01-02 2022-12-30          1008   
             Pure Inverse Volatility   2019-01-02 2022-12-30          1008   
             Composite Score           2019-01-02 2022-12-30          1008   
2023-Present Fixed 50/50 Sleeves       2023-01-03 2026-07-01           876   
             Shrunk Inverse Volatility 2023-01-03 2026-07-01           876   
             Pure Inverse Volatility   2023-01-03 2026-07-01           876   
             Composite Score           2023-01-03 2026-07-01           876   

                                        net_annualised_return  \
period       portfolio                                          
2015-2018    Fixed 50/50 Sleeves                       0.0323   
             Shrunk Inverse Volatility                 0.0281   
             Pure Inverse Volatility                   0.0254   
             Composite Score                           0.0460   
2019-2022    Fixed 50/50 Sleeves                       0.0126   
             Shrunk Inverse Volatility                 0.0299   
             Pure Inverse Volatility                   0.0406   
             Composite Score                           0.0151   
2023-Present Fixed 50/50 Sleeves                       0.2901   
             Shrunk Inverse Volatility                 0.2718   
             Pure Inverse Volatility                   0.2594   
             Composite Score                           0.3928   

                                        net_annualised_volatility  net_sharpe  \
period       portfolio                                                          
2015-2018    Fixed 50/50 Sleeves                           0.1369      0.3010   
             Shrunk Inverse Volatility                     0.1362      0.2721   
             Pure Inverse Volatility                       0.1358      0.2529   
             Composite Score                               0.1684      0.3519   
2019-2022    Fixed 50/50 Sleeves                           0.1538      0.1583   
             Shrunk Inverse Volatility                     0.1438      0.2766   
             Pure Inverse Volatility                       0.1418      0.3514   
             Composite Score                               0.2019      0.1754   
2023-Present Fixed 50/50 Sleeves                           0.2048      1.3470   
             Shrunk Inverse Volatility                     0.2028      1.2875   
             Pure Inverse Volatility                       0.2020      1.2436   
             Composite Score                               0.2560      1.4237   

                                        max_drawdown  \
period       portfolio                                 
2015-2018    Fixed 50/50 Sleeves             -0.1952   
             Shrunk Inverse Volatility       -0.1954   
             Pure Inverse Volatility         -0.1957   
             Composite Score                 -0.2669   
2019-2022    Fixed 50/50 Sleeves             -0.2566   
             Shrunk Inverse Volatility       -0.2026   
             Pure Inverse Volatility         -0.1783   
             Composite Score                 -0.3063   
2023-Present Fixed 50/50 Sleeves             -0.1711   
             Shrunk Inverse Volatility       -0.1713   
             Pure Inverse Volatility         -0.1715   
             Composite Score                 -0.2269   

                                        average_rebalance_turnover  \
period       portfolio                       

In [53]:
subperiod_comparison_metrics = [
    "net_annualised_return",
    "net_annualised_volatility",
    "net_sharpe",
    "max_drawdown",
    "average_rebalance_turnover",
    "total_transaction_cost",
    "average_daily_gross_exposure",
]

pure_minus_shrunk_rows = []

for period_name in (
    pure_subperiod_summary
    .index.get_level_values("period")
    .unique()
):
    period_summary = pure_subperiod_summary.loc[
        period_name
    ]

    difference = (
        period_summary.loc[
            "Pure Inverse Volatility",
            subperiod_comparison_metrics,
        ]
        - period_summary.loc[
            "Shrunk Inverse Volatility",
            subperiod_comparison_metrics,
        ]
    )

    difference.name = period_name
    pure_minus_shrunk_rows.append(difference)

pure_minus_shrunk_by_subperiod = pd.DataFrame(
    pure_minus_shrunk_rows
)

pure_minus_shrunk_by_subperiod.index.name = "period"

pure_minus_shrunk_by_subperiod.round(4)

,net_annualised_return,net_annualised_volatility,net_sharpe,max_drawdown,average_rebalance_turnover,total_transaction_cost,average_daily_gross_exposure
period,,,,,,,
2015-2018,-0.0027,-0.0004,-0.0193,-0.0002,0.0080,0.0012,0.0165
2019-2022,0.0107,-0.0020,0.0748,0.0243,0.0168,0.0034,0.0423
2023-Present,-0.0123,-0.0008,-0.0439,-0.0002,0.0101,0.0018,0.0174


### 6.6 Subperiod findings

Pure and shrunk inverse-volatility allocations were compared across 2016–2018, 2019–2022, and 2023–present.

Pure inverse volatility does not consistently outperform the shrunk variant. Its main advantage occurs during 2019–2022, when it increases net annualised return by 1.07 percentage points, raises net Sharpe by 0.075, and improves maximum drawdown by 2.43 percentage points. This period accounts for most of its modest full-sample Sharpe and drawdown advantage.

In 2016–2018 and 2023–present, the shrunk allocation produces higher return and Sharpe. Maximum drawdown is effectively unchanged in both periods. Pure inverse volatility also generates higher turnover, transaction costs, and gross exposure in every subperiod.

These results suggest that exact sleeve-level risk balance is not persistently superior. Its wider allocation changes were beneficial during 2019–2022 but provided no corresponding downside improvement in the other regimes. The full-sample benefit should therefore be interpreted as regime-dependent rather than structurally robust.

The shrunk inverse-volatility portfolio is retained as the preferred dynamic implementation. It regularises pure inverse-volatility weights by 40% toward equal allocation, reducing sensitivity to estimated volatility differences. Pure inverse volatility remains an important theoretical benchmark, while fixed 50/50 sleeves remain the simplest allocation baseline.

## 7. SPY benchmark comparison

### 7.1 Construct the SPY portfolio

In [54]:
from alpha_research.config.paths import (
    PROCESSED_DATA_DIR,
    RAW_DATA_DIR,
)
from alpha_research.data_loader import load_parquet


spy_raw = load_parquet(RAW_DATA_DIR / "spy_benchmark.parquet")

stock_returns = load_parquet(PROCESSED_DATA_DIR / "sp100_returns.parquet")

print("SPY raw shape:", spy_raw.shape)
print("SPY raw columns:", spy_raw.columns.tolist())
print()
print(
    "Processed stock returns shape:",
    stock_returns.shape,
)
print(
    "Processed stock returns columns:",
    stock_returns.columns.tolist(),
)

display(spy_raw.head())
display(stock_returns.head())

SPY raw shape: (2891, 8)
SPY raw columns: ['date', 'ticker', 'open', 'high', 'low', 'close', 'adj_close', 'volume']

Processed stock returns shape: (284249, 3)
Processed stock returns columns: ['date', 'ticker', 'ret_1d']


,date,ticker,open,high,low,close,adj_close,volume
0,2015-01-02,SPY,206.380005,206.880005,204.179993,205.429993,169.687881,121465900
1,2015-01-05,SPY,204.169998,204.369995,201.350006,201.720001,166.623306,169632600
2,2015-01-06,SPY,202.089996,202.720001,198.860001,199.820007,165.053894,209151400
3,2015-01-07,SPY,201.419998,202.720001,200.880005,202.309998,167.110687,125346700
4,2015-01-08,SPY,204.009995,206.160004,203.990005,205.899994,170.076111,147217800


,date,ticker,ret_1d
0,2015-01-02,AAPL,NaN
1,2015-01-05,AAPL,-0.028172
2,2015-01-06,AAPL,0.000094
3,2015-01-07,AAPL,0.014022
4,2015-01-08,AAPL,0.038422


In [55]:
required_spy_columns = {
    "date",
    "ticker",
    "adj_close",
}

missing_spy_columns = required_spy_columns - set(spy_raw.columns)

if missing_spy_columns:
    raise ValueError("Missing SPY columns: " f"{sorted(missing_spy_columns)}")

spy_source = spy_raw.copy()

spy_source["date"] = pd.to_datetime(spy_source["date"])

spy_source["ticker"] = spy_source["ticker"].astype(str).str.upper()

unexpected_tickers = set(spy_source["ticker"].dropna().unique()) - {"SPY"}

if unexpected_tickers:
    raise ValueError(
        "Unexpected tickers in the SPY file: " f"{sorted(unexpected_tickers)}"
    )

if spy_source.duplicated(subset=["date", "ticker"]).any():
    raise ValueError("Duplicate date-ticker observations found " "in the SPY dataset.")

spy_source = spy_source.sort_values("date").reset_index(drop=True)

if spy_source["adj_close"].isna().any() or spy_source["adj_close"].le(0).any():
    raise ValueError("SPY adjusted prices contain missing or " "non-positive values.")

print(
    "SPY price range:",
    spy_source["date"].min().date(),
    "to",
    spy_source["date"].max().date(),
)
print("SPY price observations:", len(spy_source))

SPY price range: 2015-01-02 to 2026-07-02
SPY price observations: 2891


In [56]:
spy_source["ret_1d"] = spy_source["adj_close"].pct_change(fill_method=None)

spy_source["forward_ret_1d"] = spy_source["ret_1d"].shift(-1)

spy_returns = spy_source[
    [
        "date",
        "ticker",
        "ret_1d",
        "forward_ret_1d",
    ]
].copy()

display(spy_returns.head())

,date,ticker,ret_1d,forward_ret_1d
0,2015-01-02,SPY,NaN,-0.018060
1,2015-01-05,SPY,-0.018060,-0.009419
2,2015-01-06,SPY,-0.009419,0.012461
3,2015-01-07,SPY,0.012461,0.017745
4,2015-01-08,SPY,0.017745,-0.008014


In [57]:
alignment_check = pd.DataFrame(
    {
        "date": spy_source["date"].iloc[:-1].to_numpy(),
        "forward_return": (spy_source["forward_ret_1d"].iloc[:-1].to_numpy()),
        "next_date_realised_return": (spy_source["ret_1d"].iloc[1:].to_numpy()),
    }
)

alignment_check["difference"] = (
    alignment_check["forward_return"] - alignment_check["next_date_realised_return"]
)

print(
    "Maximum alignment difference:",
    alignment_check["difference"].abs().max(),
)

display(alignment_check.head())

Maximum alignment difference: 0.0


,date,forward_return,next_date_realised_return,difference
0,2015-01-02,-0.018060,-0.018060,0.0
1,2015-01-05,-0.009419,-0.009419,0.0
2,2015-01-06,0.012461,0.012461,0.0
3,2015-01-07,0.017745,0.017745,0.0
4,2015-01-08,-0.008014,-0.008014,0.0


In [58]:
spy_daily = (
    spy_returns[["date", "forward_ret_1d"]]
    .dropna(subset=["forward_ret_1d"])
    .rename(
        columns={
            "forward_ret_1d": "gross_return",
        }
    )
    .loc[lambda df: df["date"] >= common_four_portfolio_start]
    .sort_values("date")
    .reset_index(drop=True)
)

if spy_daily.empty:
    raise ValueError("No SPY returns overlap with the " "portfolio evaluation period.")

spy_daily["turnover"] = 0.0

# One initial purchase, followed by buy-and-hold.
spy_daily.loc[0, "turnover"] = 1.0

spy_daily["transaction_cost"] = (
    spy_daily["turnover"] * config.transaction_cost_bps / 10_000
)

spy_daily["net_return"] = spy_daily["gross_return"] - spy_daily["transaction_cost"]

spy_daily["turnover"] = 0.0
spy_daily["is_rebalance"] = False
spy_daily["missing_return_weight"] = 0.0

# Initial portfolio formation.
spy_daily.loc[0, "turnover"] = 1.0
spy_daily.loc[0, "is_rebalance"] = True

spy_daily["transaction_cost"] = (
    spy_daily["turnover"] * config.transaction_cost_bps / 10_000
)

spy_daily["net_return"] = spy_daily["gross_return"] - spy_daily["transaction_cost"]

spy_daily["gross_exposure"] = 1.0
spy_daily["net_exposure"] = 1.0

print("SPY start:", spy_daily["date"].min().date())
print("SPY end:", spy_daily["date"].max().date())
print("SPY observations:", len(spy_daily))
print(
    "Total transaction cost:",
    spy_daily["transaction_cost"].sum(),
)

display(spy_daily.head())

SPY start: 2016-01-07
SPY end: 2026-07-01
SPY observations: 2635
Total transaction cost: 0.001


,date,gross_return,turnover,transaction_cost,net_return,is_rebalance,missing_return_weight,gross_exposure,net_exposure
0,2016-01-07,-0.010977,1.0,0.001,-0.011977,True,0.0,1.0,1.0
1,2016-01-08,0.000990,0.0,0.000,0.000990,False,0.0,1.0,1.0
2,2016-01-11,0.008069,0.0,0.000,0.008069,False,0.0,1.0,1.0
3,2016-01-12,-0.024941,0.0,0.000,-0.024941,False,0.0,1.0,1.0
4,2016-01-13,0.016417,0.0,0.000,0.016417,False,0.0,1.0,1.0


### 7.2 Add SPY to the full-period comparison

In [59]:
portfolios_with_spy = {
    **four_portfolios,
    "SPY Buy and Hold": spy_daily,
}

comparison_start = max(daily["date"].min() for daily in portfolios_with_spy.values())

comparison_end = min(daily["date"].max() for daily in portfolios_with_spy.values())

spy_benchmark_rows = []

for portfolio_name, daily in portfolios_with_spy.items():
    evaluation_daily = daily.loc[
        daily["date"].between(
            comparison_start,
            comparison_end,
        )
    ].copy()

    gross_summary = summarise_backtest(
        evaluation_daily,
        return_column="gross_return",
    ).iloc[0]

    net_summary = summarise_backtest(
        evaluation_daily,
        return_column="net_return",
    ).iloc[0]

    spy_benchmark_rows.append(
        {
            "portfolio": portfolio_name,
            "start_date": evaluation_daily["date"].min(),
            "end_date": evaluation_daily["date"].max(),
            "observations": len(evaluation_daily),
            "net_annualised_return": net_summary["annualised_return"],
            "net_annualised_volatility": net_summary["annualised_volatility"],
            "net_sharpe": net_summary["sharpe_ratio"],
            "max_drawdown": net_summary["max_drawdown"],
            "average_rebalance_turnover": net_summary["average_rebalance_turnover"],
            "total_transaction_cost": net_summary["total_transaction_cost"],
            "average_daily_gross_exposure": (evaluation_daily["gross_exposure"].mean()),
        }
    )

spy_benchmark_summary = pd.DataFrame(spy_benchmark_rows).set_index("portfolio")

spy_benchmark_summary.round(4)

,start_date,end_date,observations,net_annualised_return,net_annualised_volatility,net_sharpe,max_drawdown,average_rebalance_turnover,total_transaction_cost,average_daily_gross_exposure
portfolio,,,,,,,,,,
Fixed 50/50 Sleeves,2016-01-07,2026-07-01,2635,0.1035,0.1684,0.6695,-0.2566,0.4355,0.2295,1.5415
Shrunk Inverse Volatility,2016-01-07,2026-07-01,2635,0.1042,0.1640,0.6866,-0.2103,0.4499,0.2371,1.5812
Pure Inverse Volatility,2016-01-07,2026-07-01,2635,0.1041,0.1629,0.6899,-0.1957,0.4619,0.2434,1.6079
Composite Score,2016-01-07,2026-07-01,2635,0.1374,0.2133,0.7106,-0.3063,0.5233,0.2758,2.0007
SPY Buy and Hold,2016-01-07,2026-07-01,2635,0.1554,0.1783,0.8998,-0.3372,1.0000,0.0010,1.0000


### 7.3 Measure dependence on SPY

In [60]:
spy_market_returns = spy_daily[["date", "gross_return"]].rename(
    columns={
        "gross_return": "spy_return",
    }
)

market_model_rows = []

for portfolio_name, daily in four_portfolios.items():
    aligned_returns = (
        daily[["date", "net_return"]]
        .rename(
            columns={
                "net_return": "portfolio_return",
            }
        )
        .merge(
            spy_market_returns,
            on="date",
            how="inner",
            validate="one_to_one",
        )
        .loc[
            lambda df: df["date"].between(
                comparison_start,
                comparison_end,
            )
        ]
        .dropna()
    )

    portfolio_array = aligned_returns["portfolio_return"].to_numpy()

    spy_array = aligned_returns["spy_return"].to_numpy()

    design_matrix = np.column_stack(
        [
            np.ones(len(spy_array)),
            spy_array,
        ]
    )

    daily_alpha, beta = np.linalg.lstsq(
        design_matrix,
        portfolio_array,
        rcond=None,
    )[0]

    fitted_returns = daily_alpha + beta * spy_array

    residuals = portfolio_array - fitted_returns

    total_sum_squares = np.sum((portfolio_array - portfolio_array.mean()) ** 2)

    residual_sum_squares = np.sum(residuals**2)

    correlation = np.corrcoef(
        portfolio_array,
        spy_array,
    )[0, 1]

    market_model_rows.append(
        {
            "portfolio": portfolio_name,
            "beta": beta,
            "annualised_market_model_alpha": (daily_alpha * 252),
            "correlation": correlation,
            "r_squared": (1.0 - residual_sum_squares / total_sum_squares),
        }
    )

strategy_market_model_summary = pd.DataFrame(market_model_rows).set_index("portfolio")

strategy_market_model_summary.round(4)

,beta,annualised_market_model_alpha,correlation,r_squared
portfolio,,,,
Fixed 50/50 Sleeves,0.4650,0.0381,0.4923,0.2423
Shrunk Inverse Volatility,0.4204,0.0451,0.4570,0.2088
Pure Inverse Volatility,0.3905,0.0497,0.4274,0.1827
Composite Score,0.6257,0.0511,0.5230,0.2735


### 7.4 Compare SPY across subperiods

In [61]:
spy_subperiod_rows = []

for period_name, (period_start, period_end) in subperiod_definitions.items():
    evaluation_start = max(
        period_start,
        comparison_start,
    )

    evaluation_end = comparison_end

    if period_end is not None:
        evaluation_end = min(
            period_end,
            comparison_end,
        )

    for portfolio_name, daily in portfolios_with_spy.items():
        period_daily = daily.loc[
            daily["date"].between(
                evaluation_start,
                evaluation_end,
            )
        ].copy()

        if period_daily.empty:
            continue

        net_summary = summarise_backtest(
            period_daily,
            return_column="net_return",
        ).iloc[0]

        spy_subperiod_rows.append(
            {
                "period": period_name,
                "portfolio": portfolio_name,
                "net_annualised_return": net_summary["annualised_return"],
                "net_annualised_volatility": net_summary["annualised_volatility"],
                "net_sharpe": net_summary["sharpe_ratio"],
                "max_drawdown": net_summary["max_drawdown"],
            }
        )

spy_subperiod_summary = pd.DataFrame(spy_subperiod_rows).set_index(
    ["period", "portfolio"]
)

spy_subperiod_summary.round(4)

net_annualised_return  \
period       portfolio                                          
2015-2018    Fixed 50/50 Sleeves                       0.0323   
             Shrunk Inverse Volatility                 0.0281   
             Pure Inverse Volatility                   0.0254   
             Composite Score                           0.0460   
             SPY Buy and Hold                          0.1105   
2019-2022    Fixed 50/50 Sleeves                       0.0126   
             Shrunk Inverse Volatility                 0.0299   
             Pure Inverse Volatility                   0.0406   
             Composite Score                           0.0151   
             SPY Buy and Hold                          0.1294   
2023-Present Fixed 50/50 Sleeves                       0.2901   
             Shrunk Inverse Volatility                 0.2718   
             Pure Inverse Volatility                   0.2594   
             Composite Score                           0.3928   
             SPY Buy and Hold                          0.2271   

                                        net_annualised_volatility  net_sharpe  \
period       portfolio                                                          
2015-2018    Fixed 50/50 Sleeves                           0.1369      0.3010   
             Shrunk Inverse Volatility                     0.1362      0.2721   
             Pure Inverse Volatility                       0.1358      0.2529   
             Composite Score                               0.1684      0.3519   
             SPY Buy and Hold                              0.1289      0.8778   
2019-2022    Fixed 50/50 Sleeves                           0.1538      0.1583   
             Shrunk Inverse Volatility                     0.1438      0.2766   
             Pure Inverse Volatility                       0.1418      0.3514   
             Composite Score                               0.2019      0.1754   
             SPY Buy and Hold                              0.2253      0.6535   
2023-Present Fixed 50/50 Sleeves                           0.2048      1.3470   
             Shrunk Inverse Volatility                     0.2028      1.2875   
             Pure Inverse Volatility                       0.2020      1.2436   
             Composite Score                               0.2560      1.4237   
             SPY Buy and Hold                              0.1517      1.4247   

                                        max_drawdown  
period       portfolio                                
2015-2018    Fixed 50/50 Sleeves             -0.1952  
             Shrunk Inverse Volatility       -0.1954  
             Pure Inverse Volatility         -0.1957  
             Composite Score                 -0.2669  
             SPY Buy and Hold                -0.1935  
2019-2022    Fixed 50/50 Sleeves             -0.2566  
             Shrunk Inverse Volatility       -0.2026  
             Pure Inverse Volatility         -0.1783  
             Composite Score                 -0.3063  
             SPY Buy and Hold                -0.3372  
2023-Present Fixed 50/50 Sleeves             -0.1711  
             Shrunk Inverse Volatility       -0.1713  
             Pure Inverse Volatility         -0.1715  
             Composite Score                 -0.2269  
             SPY Buy and Hold                -0.1876

### 7.5 Findings: comparison with SPY

A standalone SPY buy-and-hold portfolio was introduced as the external passive benchmark. Over the common evaluation period from January 2016 to July 2026, SPY generated a 15.54% annualised return and a Sharpe ratio of 0.900, outperforming all four active portfolios on both measures.

The sleeve portfolios nevertheless provided meaningful downside protection. Their maximum drawdowns ranged from -19.57% to -25.66%, compared with -33.72% for SPY. The composite portfolio also experienced a smaller drawdown than SPY, although it operated with approximately twice the average gross exposure.

Market-model regressions show that the active portfolios are not simply leveraged or repackaged market exposure. Their SPY betas range from 0.39 to 0.63, while their R-squared values range from 0.18 to 0.27. All four portfolios produce positive annualised market-model intercepts, ranging from 3.81% to 5.11%. These estimates should be interpreted as beta-adjusted market-model alpha rather than CAPM alpha, because the regression uses total rather than excess returns.

Performance is strongly regime-dependent. SPY clearly outperformed the active portfolios during both 2016-2018 and 2019-2022. The risk-balanced portfolios nevertheless reduced drawdowns substantially during 2019-2022. From 2023 onward, the active portfolios became much stronger: the composite portfolio earned 39.28% annually, compared with 22.71% for SPY, but its higher volatility left its Sharpe ratio almost identical to that of SPY.

Overall, the active portfolios did not outperform passive SPY consistently in absolute or risk-adjusted terms. Their more defensible advantages are lower market beta, positive beta-adjusted returns, and improved downside protection. The strong post-2023 results should therefore be treated as a favourable regime rather than evidence of universal superiority.

## 8. Export results

In [62]:
SLEEVE_NAMES = [
    "Momentum",
    "Realised Volatility",
]

DAILY_RESULT_COLUMNS = [
    "gross_return",
    "net_return",
    "turnover",
    "transaction_cost",
    "is_rebalance",
    "missing_return_weight",
    "gross_exposure",
    "net_exposure",
]

In [63]:
optimisation_sleeve_returns = (
    sleeve_return_frame[SLEEVE_NAMES]
    .dropna(how="any")
    .rename_axis("date")
    .reset_index()
    .sort_values("date")
    .reset_index(drop=True)
)

optimisation_sleeve_returns["date"] = pd.to_datetime(
    optimisation_sleeve_returns["date"]
)

if optimisation_sleeve_returns["date"].duplicated().any():
    raise ValueError("Duplicate dates found in sleeve returns.")

if optimisation_sleeve_returns[SLEEVE_NAMES].isna().any().any():
    raise ValueError("Missing values remain in the common " "sleeve-return history.")

sleeve_returns_path = (
    PROCESSED_DATA_DIR / "portfolio_optimisation_sleeve_returns.parquet"
)

optimisation_sleeve_returns.to_parquet(
    sleeve_returns_path,
    index=False,
)

print("Saved:", sleeve_returns_path)
print("Shape:", optimisation_sleeve_returns.shape)
print(
    "Date range:",
    optimisation_sleeve_returns["date"].min().date(),
    "to",
    optimisation_sleeve_returns["date"].max().date(),
)

display(optimisation_sleeve_returns.head())

Saved: data/processed/portfolio_optimisation_sleeve_returns.parquet
Shape: (2635, 3)
Date range: 2016-01-07 to 2026-07-01


,date,Momentum,Realised Volatility
0,2016-01-07,0.001014,-0.010974
1,2016-01-08,0.014445,-0.006201
2,2016-01-11,0.006247,0.003129
3,2016-01-12,-0.011570,-0.023755
4,2016-01-13,-0.008449,0.010670


In [64]:
optimisation_sleeve_targets = pd.concat(
    [
        (momentum_targets[["date", "ticker", "weight"]].assign(sleeve="Momentum")),
        (
            volatility_targets[["date", "ticker", "weight"]].assign(
                sleeve="Realised Volatility"
            )
        ),
    ],
    ignore_index=True,
)[["date", "ticker", "sleeve", "weight"]]

optimisation_sleeve_targets["date"] = pd.to_datetime(
    optimisation_sleeve_targets["date"]
)

optimisation_sleeve_targets = optimisation_sleeve_targets.sort_values(
    ["date", "sleeve", "ticker"]
).reset_index(drop=True)

if optimisation_sleeve_targets.duplicated(["date", "ticker", "sleeve"]).any():
    raise ValueError("Duplicate sleeve target weights found.")

if set(optimisation_sleeve_targets["sleeve"].unique()) != set(SLEEVE_NAMES):
    raise ValueError("Unexpected or missing sleeve names.")

sleeve_targets_path = (
    PROCESSED_DATA_DIR / "portfolio_optimisation_sleeve_targets.parquet"
)

optimisation_sleeve_targets.to_parquet(
    sleeve_targets_path,
    index=False,
)

print("Saved:", sleeve_targets_path)
print("Shape:", optimisation_sleeve_targets.shape)
print(
    "Rebalance dates:",
    optimisation_sleeve_targets["date"].nunique(),
)

display(optimisation_sleeve_targets.head())

Saved: data/processed/portfolio_optimisation_sleeve_targets.parquet
Shape: (113656, 4)
Rebalance dates: 578


,date,ticker,sleeve,weight
0,2015-01-02,AAPL,Momentum,0.0
1,2015-01-02,ABBV,Momentum,0.0
2,2015-01-02,ABT,Momentum,0.0
3,2015-01-02,ACN,Momentum,0.0
4,2015-01-02,ADBE,Momentum,0.0


In [65]:
optimisation_benchmarks = pd.concat(
    [
        (
            daily.loc[
                daily["date"].between(
                    comparison_start,
                    comparison_end,
                ),
                ["date", *DAILY_RESULT_COLUMNS],
            ].assign(portfolio=portfolio_name)
        )
        for portfolio_name, daily in portfolios_with_spy.items()
    ],
    ignore_index=True,
)[
    [
        "date",
        "portfolio",
        *DAILY_RESULT_COLUMNS,
    ]
]

optimisation_benchmarks["date"] = pd.to_datetime(optimisation_benchmarks["date"])

optimisation_benchmarks = optimisation_benchmarks.sort_values(
    ["portfolio", "date"]
).reset_index(drop=True)

if optimisation_benchmarks.duplicated(["portfolio", "date"]).any():
    raise ValueError("Duplicate benchmark portfolio dates found.")

benchmarks_path = PROCESSED_DATA_DIR / "portfolio_optimisation_benchmarks.parquet"

optimisation_benchmarks.to_parquet(
    benchmarks_path,
    index=False,
)

print("Saved:", benchmarks_path)
print("Shape:", optimisation_benchmarks.shape)
print(
    "Portfolios:",
    optimisation_benchmarks["portfolio"].unique().tolist(),
)

Saved: data/processed/portfolio_optimisation_benchmarks.parquet
Shape: (13175, 10)
Portfolios: ['Composite Score', 'Fixed 50/50 Sleeves', 'Pure Inverse Volatility', 'SPY Buy and Hold', 'Shrunk Inverse Volatility']
